# Notebook 11 -- exploratory E3 to E7

**Asks:** run the five exploratory language analyses as one family.

**Reads** `ensemble_scores.csv` (notebook 07), `facet_dictionary_scores.csv`
(notebook 01) and `gate_results.csv` (notebook 10), plus -- where they are in the
folder -- `retribution_scored.csv` for the survey scales, `word_list_coverage.csv`
(notebook 02) and `facet_similarity.csv` (notebook 09).

**Writes** `E3_to_E7_results.csv` (every block stacked, each row carrying the gate
standing of the facets it rests on) and `fdr_exploratory_family_E3toE7.csv` (the
one corrected family; its `p_bh` column is the number to quote, never the raw `p`).

### One family, five analyses, and nothing confirmatory

Prereg 2.4: "E3 through E7 are exploratory and depend on the validation step in
Section 6.7." Prereg 6.7: "E3 through E7 form one exploratory family; we control
the false discovery rate within it and report effect sizes and confidence
intervals."

So the family is E3 + E4 + E5 + E6 + E7. **E5 and E6 are inside it**, and so is
every test produced by a descriptive-only facet (censure, moral balance,
proportionality). Holding a block out would understate the multiplicity this
analysis actually incurs, which is the one thing a correction exists to prevent.
Benjamini-Hochberg -- which controls the expected share of false findings among
the findings declared -- is applied once, across all five blocks together.

Outside the family: the prereg 6.6 non-language exploratory analyses (E1, E2, the
factor structure of the correlates, the sentence change, the political
moderators). Prereg 6.6 leaves those uncorrected in as many words: "These analyses
are exploratory and are not corrected as a confirmatory family."

### The one-row-per-participant contract is checked here

948 participants x 3 answers = 2,844 rows, keyed on (`response_id`, `item_type`).
Notebook 00 checks that for `openends_long.csv`; nothing between 00 and here
re-checks it. So this notebook checks it on the way in, in **Load** and in **Do 2**.
One duplicated word-list row raises E4's, E6's and E7's sample sizes together while
the paired-participant line goes on printing 948, and shrinks every p-value in the
corrected family. Nothing else in the chain would show it.

### The five

- **E3** ranking-defined retributivists (`Prioritization_1 == 1`) against everyone
  else, on Q333 and Q1 **separately, never pooled**. Agreement across the two
  items is the within-study replication.
- **E4** the within-person desert x revenge co-occurrence -- the claim the study
  is built around -- estimated from **separately elicited** facet scores.
- **E5** whether desert language and proportionality language travel together in
  the text: the text-side version of H6.
- **E6** the backward-minus-forward register shift, within person, all nine facets.
- **E7** whether facet language predicts the sentence beyond the eight-item
  self-report punitiveness scale.

### E4's preregistered corroborator is run, not skipped

Prereg 6.7 requires E4's co-occurrence "to show up in the non-model word-list
measure". That check runs here, with its interval, its joint-firing row count and
its realized prevalence printed beside it. Word-list revenge fires on under 1% of
responses, so the check has almost no power in either direction. That is an
observed limitation to report with its counts -- never disconfirmation of E4, and
never an invitation to read a near-empty measure as a result.

### The gate travels on every row

Facets that did not clear the notebook 10 gate are still analysed and still
exported: dropping them would make the gate invisible to anyone reading the
results. Every row therefore carries the standing of the facets it rests on, so a
descriptive-only facet's number cannot be read as a sound one by anybody who
opens the file. Gate-backed is a statement about a FACET's validation standing,
never about a TEST's confirmatory status.

## Config

In [ ]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "11"                     # this notebook's number
NB_NAME     = "exploratory_E3_E7"      # this notebook's job, in two words
NB_ASKS     = "Run the five exploratory language analyses as one family."

## Setup

In [ ]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

In [ ]:
# === SETUP (cont.): imports and display width ===
# WHAT THIS CELL DOES: imports the four libraries the five analyses need and
# widens the printed tables. It computes nothing.
#
# NOTHING IS INSTALLED HERE. pandas, numpy, scipy and statsmodels all ship with
# the Colab image, and every number below comes out of those four.
#   scipy        Welch's t (E3), the one-sample t (E6), Pearson correlations
#                (E4, E5, E7) and the normal and F distributions the remaining
#                p-values are read off.
#   statsmodels  the two nested regressions E7 compares, and the Benjamini-
#                Hochberg correction the whole family goes through in one call.
#
# WHY THE DISPLAY WIDTH MATTERS: these tables are wide (block x estimate x
# interval x n x gate standing). At the default width pandas replaces the middle
# columns with an ellipsis, and a truncated results table is worse than none: a
# number that never reaches the screen still reads as though it had been checked.
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, ttest_ind, ttest_1samp, norm, f as f_dist
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import warnings

# WARNINGS ARE NOT SILENCED GLOBALLY, and that is deliberate. A bare
# warnings.filterwarnings("ignore") switches off every warning for the remainder of
# the runtime, and the warnings it would switch off include the ones this notebook
# most needs a reader to see: scipy raises ConstantInputWarning when a correlation is
# asked for on a column that never varies, which is precisely the state E4's sparse
# word-list corroborator and E5's proportionality column can be in, and a correlation
# on a column that never varies is not a null result about co-occurrence.
# Narrowing the filter to category=RuntimeWarning would NOT be enough either:
# scipy's ConstantInputWarning is itself a subclass of RuntimeWarning, so that filter
# hides it too. Suppression is therefore scoped, in DO 5, to the one place where
# degenerate input is EXPECTED rather than informative -- the bootstrap's resample
# loop, where a handful of all-identical draws out of 5,000 is normal and would
# otherwise print thousands of identical lines over the tables. Everything outside
# that loop warns normally, including every estimate computed on the observed sample.

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
np.set_printoptions(precision=4, suppress=True)

print("Imports ready: pandas, numpy, scipy, statsmodels. Nothing to install.")

In [ ]:
# === SETUP (cont.): the facet vocabulary, the three items, and the one family ===
# The nine facets in prereg 5.15's order: four principled-retributive (desert,
# proportionality, censure, moral_balance), two for retribution's harder edge
# (revenge, suffering), then the three consequentialist goals, which are scored so
# retributive language has a full menu of punishment justifications to be measured
# against and are NEVER summed into any retributivism total.
FACETS = ["desert", "proportionality", "censure", "moral_balance",
          "revenge", "suffering", "deterrence", "incapacitation", "rehabilitation"]

# ranking = Q333, the goal-ranking explanation; backward = Sentence_OpenEnd1, the
# sentence justification; forward = Sentence_OpenEnd2, the accomplishment item.
ITEM_TYPES = ["ranking", "backward", "forward"]

# E6 predicts a SIGN PATTERN off this split: the six retributive facets peak on the
# backward item, the three consequentialist goals on the forward one. The class is
# about the CONSTRUCT, not about the facet's gate standing -- proportionality,
# censure and moral_balance are retributive here even though all three are routed
# descriptive-only.
RETRIBUTIVE = ["desert", "proportionality", "censure", "moral_balance",
               "revenge", "suffering"]
CONSEQUENTIALIST = ["deterrence", "incapacitation", "rehabilitation"]

# THE TWO LISTS MUST PARTITION THE NINE FACETS: every facet in exactly one of them.
# E6 reads BOTH lists to decide a facet's predicted direction. A facet in neither
# would otherwise be handed the consequentialist prediction by default and have its
# expected sign quietly inverted; a facet in both would be classed by whichever test
# ran first. This fires here, at the top of the notebook, rather than surfacing forty
# cells later as a wrong sign in a results table nobody can check by eye.
_unclassified = [f for f in FACETS
                 if (f in RETRIBUTIVE) == (f in CONSEQUENTIALIST)]
assert not _unclassified, (
    "every facet must appear in exactly one of RETRIBUTIVE or CONSEQUENTIALIST, "
    "because E6 reads its predicted sign off them. In neither list, or in both: "
    f"{_unclassified}")

# THE FAMILY TAG, and why it travels on the row rather than living in a comment.
# Prereg 2.4 and 6.7 put E3, E4, E5, E6 and E7 in ONE exploratory family with the
# false discovery rate controlled within it. Every hypothesis test below is stamped
# with this string, and the correction cell selects on it, so a number's standing
# cannot be separated from the number when this table is read or quoted.
FAMILY = "exploratory_E3toE7"
BLOCKS = ["E3", "E4", "E5_H6", "E6", "E7"]

# RUN_MODE = "smoke" is a plumbing check, not a measurement. This notebook trims
# nothing itself -- the smoke corpus was trimmed by PARTICIPANT back in notebook 00,
# so every person still carries all three of their answers and the within-person
# tests (E6's difference, E7's complete cases) are still computable. What a
# twenty-participant corpus cannot support is any of the numbers below.
if RUN_MODE == "smoke":
    print("RUN_MODE = 'smoke': the tables below prove the plumbing works.")
    print("They are NOT results. Re-run the chain in 'full' before reading any.")

print(f"Facets: {len(FACETS)}. Items: {', '.join(ITEM_TYPES)}.")
print(f"One BH-FDR family across {', '.join(BLOCKS)} (prereg 2.4, 6.7).")

## Load

In [ ]:
# === LOAD (1 of 2): the three files this notebook cannot run without ===
# WHAT THIS CELL DOES: fetches the required inputs through need(), which either
# hands back a path or stops with a sentence saying which notebook makes the file.
#
# CONSUMES
#   ensemble_scores.csv          notebook 07. Long: one row per response x item x
#                                facet, carrying the ensemble mean of record. It
#                                also carries EXTRA rows for the separately
#                                elicited E4 pair, under facet = 'desert_solo' and
#                                'revenge_solo'. Those rows are E4's primary
#                                estimate and are kept here rather than filtered
#                                out -- the gate notebook sets them aside because
#                                the gate is applied to the joint-call score of
#                                record, but E4 is the one analysis that needs
#                                them.
#   facet_dictionary_scores.csv  notebook 01. The word-list stream, which is E4's
#                                preregistered non-model corroborator, and the only
#                                place n_words, vignette and orientation travel
#                                from.
#   gate_results.csv             notebook 10. Which facets are gate-backed, which
#                                are descriptive-only, and why. Every result row
#                                below is stamped with it.
ENS_PATH  = need("ensemble_scores.csv", produced_by="07")
DICT_PATH = need("facet_dictionary_scores.csv", produced_by="01")
GATE_PATH = need("gate_results.csv", produced_by="10")

ens     = pd.read_csv(ENS_PATH)
dict_df = pd.read_csv(DICT_PATH)
gate    = pd.read_csv(GATE_PATH)

# THE KEY CONTRACT, CHECKED ON THE WAY IN RATHER THAN ASSUMED. 948 participants x 3
# items = 2,844 rows keyed on (response_id, item_type). Notebook 00 checks that for
# openends_long.csv and nothing between 00 and here re-checks it, so this notebook
# checks every keyed file it reads. Both files below are REQUIRED, so a defect in one
# is a broken pipeline rather than a missing extra and it stops the run here, naming
# the notebook to re-run. The OPTIONAL inputs are handled the other way further down:
# they print what is wrong and the run carries on without them.
_dup_ens = int(ens.duplicated(["response_id", "item_type", "facet"]).sum())
assert _dup_ens == 0, (
    f"ensemble_scores.csv holds {_dup_ens} duplicated (response_id, item_type, "
    "facet) row(s). The pivot in DO 1 would AVERAGE them into a single cell and "
    "hand back a table that looks correct, and the mean of a row written twice is "
    "not the ensemble score of record. Re-run notebook 07.")
# The gate table settles the standing stamped on every result row below, and the
# lookup built in DO 4 keeps the LAST row it sees for a facet. Two rows for one facet
# would let file order decide whether that facet's results read as gate-backed.
_dup_gate = int(gate.duplicated(["facet"]).sum())
assert _dup_gate == 0, (
    f"gate_results.csv holds {_dup_gate} duplicated facet row(s). Each facet must "
    "carry exactly one gate verdict. Re-run notebook 10.")

# The separate-call rows, read off the file rather than assumed. Notebook 06 writes
# them only when the split-call elicitation ran, and E4's primary estimate depends
# on them, so their presence or absence is reported here at the top rather than
# discovered in the middle of E4.
SOLO_ROWS = sorted(set(ens["facet"].unique()) - set(FACETS))
print(f"  ensemble rows:   {len(ens):,}  ({ens.response_id.nunique():,} responses)")
print(f"  dictionary rows: {len(dict_df):,}")
print(f"  gate table:      {len(gate)} facet rows")
print("  separate-call rows present: "
      + (", ".join(SOLO_ROWS) if SOLO_ROWS else "NONE -- see the E4 cells below"))

In [ ]:
# === LOAD (2 of 2): the optional inputs, and what each absence costs ===
# NONE of these stops the notebook. Each absence prints one line, changes exactly
# one thing below, and is reported in the results file as "not run" rather than
# left blank. A blank cell in a results table cannot be told apart from a measured
# zero, and the two mean opposite things.
#
#   retribution_scored.csv  the survey scaffold from the R pipeline, N = 948 after
#                           the preregistered 6.2 exclusions. It carries
#                           Prioritization_1 (E3's grouping variable), the survey
#                           scales E4 and E5 are benchmarked against, and
#                           Sentence1_z and punitiveness (E7's whole model).
#                           Without it E3, E5's survey side and E7 do not run.
#   word_list_coverage.csv  notebook 02. Says whether the word list fires often
#                           enough to corroborate anything (the 5% floor). The
#                           verdict is CARRIED from that notebook, never recomputed
#                           here: a rule applied in two places is a rule whose two
#                           copies eventually disagree.
#   facet_similarity.csv    notebook 09. The embedding stream, which is EXCLUDED
#                           from every estimate in this notebook by design (the
#                           reference pools are built from disjoint per-facet seeds,
#                           so joint firing is depressed mechanically). It is read
#                           only so that exclusion can be shown as a number rather
#                           than asserted.
#   gold_codes.csv          the optional human-coding notebook, and only when
#                           HUMAN_GOLD is True. Prereg 6.7 has the model ensemble
#                           code the corpus IN PLACE OF human raters and prescribes
#                           no human step, so the human arm is a DEVIATION and
#                           nothing in this notebook depends on it.
SURV_PATH = need("retribution_scored.csv", produced_by="00", optional=True)
COV_PATH  = need("word_list_coverage.csv", produced_by="02", optional=True)
SIM_PATH  = need("facet_similarity.csv", produced_by="09", optional=True)
GOLD_PATH = (need("gold_codes.csv", produced_by="the optional human-coding notebook",
                  optional=True) if HUMAN_GOLD else None)
if not HUMAN_GOLD:
    print("  [skip] the human arm: HUMAN_GOLD is False, so no human coding exists")
    print("         anywhere in this chain. Every human corroborator below reads")
    print("         'not run' -- never blank, and never a failed check.")

survey   = pd.read_csv(SURV_PATH) if SURV_PATH else None
coverage = pd.read_csv(COV_PATH) if COV_PATH else None
sim      = pd.read_csv(SIM_PATH) if SIM_PATH else None
gold     = pd.read_csv(GOLD_PATH) if GOLD_PATH else None

## Do

In [ ]:
# === DO (1 of 24): the analysis frame, part 1 -- the ensemble scores ===
# WHAT THIS CELL DOES: turns the long ensemble file into one row per response x
# item, with one column per facet.
#
# THE NAMESPACE RULE, AND WHY IT IS THE MOST IMPORTANT LINE IN THIS NOTEBOOK. Six
# of the nine facet names are ALSO survey scale names (revenge, suffering,
# proportionality, deterrence, incapacitation, rehabilitation). A bare facet column
# and a survey column would land on the same name in the merge, and pandas would
# settle it with _x/_y suffixes on whichever frame arrived second. One prefix per
# source keeps every name unambiguous:
#     ens_<facet>     the language-model ensemble mean          (notebook 07)
#     dict_<facet>    the word-list hit count                   (notebook 01)
#     <bare name>     a participant-level survey scale          (retribution_scored)
# E4 is the analysis this protects: reading ens_ where the word list is meant would
# turn the independent non-model corroborator into the ensemble correlating with
# itself, and reading the bare names would correlate two SURVEY scales and report
# the result as if it had come from the text.
ens_wide = (ens.pivot_table(index=["response_id", "item_type"], columns="facet",
                            values="ensemble_mean").reset_index())
ens_wide.columns = (["response_id", "item_type"]
                    + [f"ens_{c}" for c in ens_wide.columns[2:]])

# The separate-call scores arrive as their own facet rows and therefore pivot into
# their own columns: ens_desert_solo and ens_revenge_solo. They are kept DISTINCT
# from ens_desert and ens_revenge on purpose -- one is E4's halo-resistant primary
# estimate, the other is the preregistered joint-call score, and the difference
# between them is the diagnostic. A single shared name would make them
# interchangeable and destroy the diagnostic.
#
# pivot_table returns exactly one row per (response_id, item_type), which is what
# E6's backward-minus-forward subtraction needs. It gets there by AVERAGING any
# duplicated (response_id, item_type, facet) rows, which would hide such a row rather
# than fix it -- so the LOAD cell asserts there are none before this line runs. The
# contract is enforced IN THIS NOTEBOOK, at that assert and again at the word-list
# merge in DO 2. It is not enforced upstream: no notebook after 00 checks it.
HAS_SOLO = [c for c in ["ens_desert_solo", "ens_revenge_solo"] if c in ens_wide.columns]

# The holistic call repeats down a response's nine facet rows in the long file, so
# one row per response is taken. It is DESCRIPTIVE COVERAGE ONLY and barred from
# E4's evidence set (design section 9): same pass, same rater as the facet vector.
if "holistic" in ens.columns:
    _hol = ens.drop_duplicates(["response_id", "item_type"])[
        ["response_id", "item_type", "holistic"]]
    ens_wide = ens_wide.merge(_hol, on=["response_id", "item_type"], how="left")

print(f"  ensemble wide: {ens_wide.shape[0]:,} rows x {ens_wide.shape[1]} columns")
print("  separate-call columns: " + (", ".join(HAS_SOLO) if HAS_SOLO else "NONE"))

In [ ]:
# === DO (2 of 24): the analysis frame, part 2 -- the word list ===
# The word list takes its prefix before it can meet a survey column. n_words,
# vignette and orientation keep their own bare names, because they are needed under
# those names: n_words is the covariate prereg 6.7 applies across the family,
# vignette is the randomly assigned scenario, and orientation is what makes an item
# backward-looking or forward-looking. All three are PER-ITEM quantities and this is
# the per-item source, so where the survey file carries its own participant-level
# copy of one of them, the next cell drops the survey's copy rather than suffixing it.
_ren = {}
for f in FACETS:
    if f in dict_df.columns:
        _ren[f] = f"dict_{f}"
    if f"{f}_per100" in dict_df.columns:
        _ren[f"{f}_per100"] = f"dict_{f}_per100"
dict_keep = dict_df.rename(columns=_ren)

# The negation-zeroed copies are dropped here on purpose. Design section 6:
# negation handling is zero-out only, whitelist-based, and EXCLUDED FROM EVERY
# GATE. The raw counts are what E4's preregistered corroborator is computed on, so
# the two versions can never be confused for one another in this notebook.
dict_keep = dict_keep[[c for c in dict_keep.columns if not c.endswith("_negzeroed")]]


In [ ]:
# === DO (2 of 24, cont.): the word list joins the ensemble ===
# THE CONTRACT, ENFORCED HERE, ON THE FRAME ABOUT TO BE MERGED IN. One duplicated
# row in facet_dictionary_scores.csv fans that participant out into two analysis
# rows, and nothing downstream would show it: E4's n, every E6 facet n and E7's
# complete-case count would all rise together while E6's paired-participant line went
# on printing 948, and every p-value in the corrected family would shrink because the
# same person had been counted twice. Notebook 01 does not assert this on export.
_dup_dict = int(dict_keep.duplicated(["response_id", "item_type"]).sum())
assert _dup_dict == 0, (
    f"facet_dictionary_scores.csv holds {_dup_dict} duplicated (response_id, "
    "item_type) row(s). Expect one row per participant per item -- 948 x 3 = 2,844 "
    "in the full corpus. Re-run notebook 01.")

_n_before = len(ens_wide)
feat = ens_wide.merge(dict_keep, on=["response_id", "item_type"], how="left",
                      suffixes=("", "_dic"))
# THE MERGE'S OWN CONTRACT: it may attach columns, never rows. This is the one merge
# in the notebook that can fan out, and the check above is what fires first and names
# the count. This line is what fires if the frame is ever fanned out by some route
# that check does not cover.
assert len(feat) == _n_before, (
    f"the word-list merge changed the row count: {_n_before} -> {len(feat)}. The "
    "word-list frame must hold at most one row per (response_id, item_type).")
# A _dic suffix means a word-list column landed on a name the ensemble already
# held, which would leave every later reference reading whichever one arrived
# first. Asserted rather than trusted, because the symptom is a plausible number
# rather than an error.
_clash = [c for c in feat.columns if c.endswith("_dic")]
assert not _clash, f"a word-list column collided with an ensemble column: {_clash}"

# A left merge keeps every ensemble row whether or not a word-list row was found, so
# two files keyed on different id spaces produce a full table of empty word-list
# columns rather than an error -- and E4's PREREGISTERED corroborator would then be
# computed on nothing at all. The match count is read off n_words, which only the
# word-list file supplies.
_matched_wl = int(feat["n_words"].notna().sum()) if "n_words" in feat.columns else -1
if _matched_wl == 0:
    print("  STOP AND READ: no ensemble row found a word-list row. The two files are")
    print("  keyed on different id spaces, so every dict_ column is empty and E4's")
    print("  preregistered word-list corroborator has nothing to run on.")

print(f"  ensemble + word list: {feat.shape[0]:,} rows x {feat.shape[1]} columns")
for _it in ITEM_TYPES:
    print(f"    {_it:9s} {int((feat['item_type'] == _it).sum()):,} rows")


In [ ]:
# === DO (3 of 24): the analysis frame, part 3 -- can the survey be attached? ===
# The survey is participant-level: one row per person fans out onto that person's
# ranking, backward and forward rows. That is intended -- E3's grouping variable and
# E7's scales are person constants -- but it is why the survey side is checked before
# it is merged, and it is why the participant-level quantities (E4's survey reference,
# E5's survey side) are computed on `surv` directly rather than on the item table.
# Computing them there would enter every participant three times.
SURVEY_OK, surv, SURVEY_ATTACHABLE = False, None, False
if survey is not None:
    # The participant id column in the R export is ResponseId. Renamed rather than
    # merged on two names, so the join key is one column everywhere in the chain.
    if "ResponseId" in survey.columns and "response_id" not in survey.columns:
        survey = survey.rename(columns={"ResponseId": "response_id"})
    # ONE ROW PER PARTICIPANT, or the merge below would enter a person's three items
    # more than once and every survey-side number would rest on duplicated rows.
    #
    # THIS IS A CHECK AND NOT A STOP, and the difference is the point.
    # retribution_scored.csv is an OPTIONAL input: E4's word-list corroborator, E4's
    # halo diagnostic and the whole of E6 never touch it. Killing the notebook over a
    # defect in a file that most of it does not read would take down analyses that had
    # nothing to do with the defect. So it is printed, the survey side is recorded as
    # NOT RUN, and the run continues -- the same way the collision guard in the next
    # cell behaves, and the same way the LOAD cell promises the optional inputs behave.
    _dup_surv = int(survey.duplicated(["response_id"]).sum())
    if _dup_surv:
        print(f"  STOP AND READ: retribution_scored.csv holds {_dup_surv} duplicated")
        print("  participant row(s). It must hold exactly one row per participant.")
        print("  E3, E5's survey side and E7 are recorded as NOT RUN rather than")
        print("  computed on duplicated rows. Everything that does not need the")
        print("  survey still runs below. Fix the export and re-run for the rest.")
    else:
        SURVEY_ATTACHABLE = True
else:
    print("  No retribution_scored.csv. E3 (needs Prioritization_1), E5's survey")
    print("  side and E7 (needs punitiveness and Sentence1_z) are recorded as NOT")
    print("  RUN. That is a missing input, not a null result.")


In [ ]:
# === DO (3 of 24, cont.): attach the survey to the item table ===
if SURVEY_ATTACHABLE:
    # The carrier columns arrive from BOTH files -- the word list carries them per
    # item, the survey per participant -- and they are the same variables, not a name
    # clash. The per-item copies are the ones the analyses need (a person's three
    # answers differ in length), so the survey's copies are dropped before the merge.
    # Dropping them here is what keeps the guard below meaningful: after this, any
    # collision left is a survey SCALE losing its bare name, which is the failure that
    # guard exists to catch.
    _carriers = [c for c in ["orientation", "vignette", "n_words", "item_type",
                             "text", "holistic"] if c in survey.columns
                 and c in feat.columns]
    if _carriers:
        print(f"  survey copies of {', '.join(_carriers)} dropped: the per-item "
              "copies from the word-list file are the ones used.")
        survey = survey.drop(columns=_carriers)
    # NO ROW-COUNT CHECK ON THIS MERGE, deliberately. A left merge onto a right frame
    # already shown unique on its key cannot change the row count, so a check here
    # could not be made to fire by any input -- and a guard that cannot fire reads
    # like protection this table does not have. The merge in this notebook that CAN
    # fan out is the word-list merge in DO 2, and the row-count check lives there.
    feat = feat.merge(survey, on="response_id", how="left", suffixes=("", "_surv"))

    # THE COLLISION GUARD. A _surv-suffixed column means a survey scale lost its bare
    # name to a text-derived column already in the table, and every later reference to
    # that bare name would read the text column instead. E4's survey reference and
    # E5's survey side would then go on printing correlations while quietly comparing
    # the ensemble against a word list.
    _collided = [c for c in feat.columns if c.endswith("_surv")]
    if _collided:
        print("  STOP AND READ: survey columns collided with text-derived columns "
              f"and were renamed: {_collided}")
        print("  Every survey-side quantity below is recorded as NOT RUN rather")
        print("  than computed against the wrong column. Fix the prefixes and re-run.")
    else:
        SURVEY_OK, surv = True, survey
        _matched = feat["response_id"].isin(set(survey["response_id"])).sum()
        print(f"  survey attached: {_matched:,} of {len(feat):,} item rows carry a "
              f"participant match ({len(survey)} participants, expect 948)")
        if _matched < 0.5 * len(feat):
            print("  WARNING: fewer than half the rows matched. The two files are")
            print("  probably keyed on different id spaces, and every survey-side")
            print("  number below would then rest on a small, non-random subset.")


In [ ]:
# === DO (4 of 24): the gate table, and what it is allowed to decide ===
# WHAT THIS CELL DOES: reads gate_results.csv into a lookup. It re-decides nothing.
# The gate is settled once, in notebook 10, and a rule applied in two places is a
# rule whose two copies eventually disagree.
#
# WHY EVERY RESULT ROW BELOW IS STAMPED WITH IT. A facet that failed the gate is
# still scored, still analysed and still exported: a results file containing only
# what passed makes the gate invisible to anyone reading the data. What must never
# happen is a failed facet turning up in an E-result that reads exactly like a
# sound one. So the standing travels ON THE ROW, and the number and its warrant
# cannot be separated when the table is opened, sorted or pasted into a draft.
#
# GATE-BACKED IS NOT CONFIRMATORY. Prereg 2.4 puts all of E3-E7 in one exploratory
# family. The gate decides whether a facet is PRIMARY or DESCRIPTIVE inside that
# family and nothing else. No verdict it can return makes a test confirmatory.
def _word(v, default):
    """One gate cell as a word, so an empty cell can never read as a verdict.

    A blank in a clause column means nobody evaluated that clause, which is not the
    same as failing it. Left alone it would arrive here as the float NaN and print
    as the string 'nan' beside a result.
    """
    s = "" if v is None else str(v).strip()
    return default if s in ("", "nan", "NaN", "None") else s

STANDING = {}
for _, r in gate.iterrows():
    STANDING[str(r["facet"])] = {
        "status":  _word(r.get("status"), "UNKNOWN"),
        "clauseA": _word(r.get("clauseA"), "not run"),
        "clauseB": _word(r.get("clauseB"), "not run"),
        "clauseC": _word(r.get("clauseC"), "not run"),
        "reason":  _word(r.get("reason"), ""),
    }
GATE_BACKED_FACETS = [f for f in FACETS
                      if STANDING.get(f, {}).get("status") == "GATE_BACKED"]
# The gate-ELIGIBLE set is read off the gate table too. It is the fallback E3 and E7
# run on when nothing cleared the gate, and hard-coding it here would let this
# notebook and notebook 10 drift apart on which facets were even candidates.
if "gate_eligible" in gate.columns:
    GATE_ELIGIBLE = gate.loc[gate["gate_eligible"].astype(str).str.strip().str.lower()
                             == "yes", "facet"].tolist()
else:
    GATE_ELIGIBLE = ["desert", "revenge", "suffering"]

# E5's null-by-absence verdict is CARRIED from notebook 10, where the realized
# proportionality base rate was computed. Below one response in ten there are too
# few proportionality-positive cases for the desert-versus-proportionality
# dissociation to have anywhere to appear.
PROP_NULL_BY_ABSENCE = (bool(gate["prop_null_by_absence"].iloc[0])
                        if "prop_null_by_absence" in gate.columns and len(gate)
                        else None)

print("Gate standing carried in from notebook 10:")
for f in FACETS:
    s = STANDING.get(f, {})
    print(f"  {f:16s} {s.get('status', 'NOT IN GATE TABLE'):18s} "
          f"A={s.get('clauseA', '--'):8s} B={s.get('clauseB', '--'):8s} "
          f"C={s.get('clauseC', '--'):8s}")
print(f"\n  gate-backed (primary standing): "
      f"{', '.join(GATE_BACKED_FACETS) if GATE_BACKED_FACETS else 'none'}")
print(f"  gate-eligible (could have been promoted): {', '.join(GATE_ELIGIBLE)}")

In [ ]:
# === DO (4 of 24, cont.): the two functions that stamp the standing on a row ===
# Every block below writes a `rests_on` column naming the facets its number is
# built from, and these two turn that into words at the end. Two columns rather
# than one, because they answer different questions: facet_standing is the audit
# trail, gate_status is what a reader skimming the file sees first.
def standing_of(facets):
    """Name every facet a row rests on, and how each one stood at the gate."""
    fs = [f for f in facets if f]
    if not fs:
        # A row that rests on no facet at all -- a survey-side reference, say. It is
        # said in words rather than left blank, because a blank in this column
        # would read as a facet whose standing nobody checked. The words avoid
        # "n/a" and "NA" on purpose: pandas reads both of those back from a CSV as
        # missing, so the sentence would vanish the moment the file was re-opened.
        return "not applicable -- this row rests on no facet"
    return "; ".join(f"{f}={STANDING.get(f, {}).get('status', 'UNKNOWN')}"
                     for f in fs)

def gate_status_of(facets):
    """The one-word summary: is every facet behind this number gate-backed?"""
    fs = [f for f in facets if f]
    if not fs:
        return "not applicable"
    st = [STANDING.get(f, {}).get("status", "UNKNOWN") for f in fs]
    if all(s == "GATE_BACKED" for s in st):
        return "all gate-backed"
    if any(s == "GATE_BACKED" for s in st):
        # The dangerous case, and the reason this is a word and not a boolean. A
        # desert x revenge correlation where only one of the two cleared the gate is
        # not a gate-backed result, and it must not read as one.
        return "MIXED -- not every facet is gate-backed"
    return "NONE gate-backed -- descriptive only"

def rests(*facets):
    """The comma string the blocks below write into `rests_on`."""
    return ",".join(f for f in facets if f)

print("Every result row below will carry rests_on, facet_standing and gate_status.")

In [ ]:
# === DO (5 of 24): the bootstrap, which every interval below comes from ===
# WHY RESAMPLING RATHER THAN A FORMULA. Almost every quantity here is computed on a
# small or badly imbalanced slice: the ranking-defined retributivists are a fraction
# of the sample, and several facets fire on a few per cent of responses. The
# analytic interval for a correlation assumes a bivariate normal shape that 0-to-1
# facet scores do not have, and there is no closed form at all for the interval
# around Cohen's d on two unequal groups. Resampling the observed rows makes no
# distributional claim: it asks what this statistic would have looked like had we
# drawn a different sample of the same people, and reads the answer off the spread.
RNG = np.random.default_rng(RANDOM_SEED)

# 5000 resamples puts 125 draws in each 2.5% tail, which holds a percentile bound
# steady to about the third decimal.
N_BOOT = 5000 if RUN_MODE == "full" else 400

# HAZARD: RNG is a single generator shared by every bootstrap below. The numbers are
# reproducible only when the cells are run in order, top to bottom, from a fresh
# runtime. Re-running one block on its own advances the stream and shifts the bounds
# in the third decimal.

def boot_ci(stat_fn, *arrays, n_boot=None, alpha=0.05):
    """Paired bootstrap interval for any statistic. Returns (estimate, lo, hi).

    Rows are resampled JOINTLY -- one index vector is drawn and applied to every
    array -- so a response keeps its desert score, its revenge score, its group
    flag and its survey answer together. Drawing each array independently would
    destroy exactly the association being estimated.

    The point estimate is computed on the observed rows, never as the mean of the
    resamples, so the reported number is the sample statistic and the interval is
    reported around it.

    A degenerate resample (every drawn row identical, so there is no variance and
    the statistic is undefined) is recorded as NaN and dropped. On the sparse
    facets a handful of those is expected, and an interval computed from the
    survivors is more honest than no interval at all.
    """
    n_boot = N_BOOT if n_boot is None else n_boot
    arrays = [np.asarray(a, dtype=float) for a in arrays]
    n = len(arrays[0])
    # Unequal lengths mean the arrays are no longer row-aligned, so the pairing this
    # function exists to preserve is already broken and the interval would be drawn
    # from mismatched people.
    assert all(len(a) == n for a in arrays), "boot_ci: array length mismatch"
    # Computed OUTSIDE the warning filter below, so a warning about the OBSERVED
    # sample -- scipy's ConstantInputWarning on a facet column that never varies, say
    # -- still reaches the screen. On these sparse facets that warning is a result.
    point = stat_fn(*arrays)
    if n < 3:
        return point, np.nan, np.nan
    stats = np.empty(n_boot)
    # The only suppression in this notebook, scoped to the resample loop and to
    # RuntimeWarning. Degenerate RESAMPLES are expected here -- a few all-identical
    # draws out of 5,000 on a facet that fires on 1% of responses -- and each one
    # would otherwise print, burying the tables under thousands of identical lines.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for b in range(n_boot):
            idx = RNG.integers(0, n, n)      # ONE index vector: this is the pairing
            try:
                stats[b] = stat_fn(*[a[idx] for a in arrays])
            except Exception:
                stats[b] = np.nan
        lo, hi = np.nanpercentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lo, hi

print(f"Bootstrap ready: {N_BOOT:,} resamples, seed {RANDOM_SEED} (frozen).")

In [ ]:
# === DO (5 of 24, cont.): the small helpers every block shares ===
def num(v, nd=3):
    """A number, or two dashes where there is none.

    Every printed table below goes through this. A quantity that could not be
    computed shows as -- rather than as an empty cell, because an empty cell in a
    results table reads as a measured zero and the two mean opposite things.
    """
    return "--" if (v is None or (isinstance(v, float) and np.isnan(v))) else f"{v:.{nd}f}"

def _corr_ci(x, y):
    """Pearson r with a paired bootstrap interval. Returns (r, lo, hi, n).

    Pearson r is the ordinary correlation coefficient: it runs from -1 to +1 and
    measures how closely two quantities rise and fall together along a straight line.

    Tolerant of the sparse columns E4 hands it. Three complete rows and nonzero
    variance on both sides, or no number at all: on the word-list columns the
    zero-variance branch is the live one, because a facet that never fires has no
    correlation to report and a 0.0 there would read as "measured, and null".
    """
    m = pd.DataFrame({"x": pd.to_numeric(x, errors="coerce"),
                      "y": pd.to_numeric(y, errors="coerce")}).dropna()
    if len(m) < 3 or m["x"].std() == 0 or m["y"].std() == 0:
        return np.nan, np.nan, np.nan, len(m)
    est, lo, hi = boot_ci(lambda a, b: pearsonr(a, b)[0],
                          m["x"].values, m["y"].values)
    return est, lo, hi, len(m)

def _r_pvalue(r, n):
    """Two-sided p for a correlation, via Fisher's z transform.

    Fisher's z transform stretches a correlation, whose possible values are penned in
    between -1 and +1, onto an unbounded scale where its sampling distribution is
    close to the normal bell curve -- which is what lets a p-value be read off the
    normal curve below.

    Needed because E4 and the E7 secondaries are estimated with bootstrap intervals
    rather than a parametric test, and Benjamini-Hochberg needs a p-value from every
    member of the family.

    Guards: n < 4 leaves no degrees of freedom after the -3, and |r| >= 1 makes the
    transform infinite. Both return NaN, so the row drops out of the family instead
    of entering it with a fabricated p.
    """
    if r is None or (isinstance(r, float) and np.isnan(r)) or n is None:
        return np.nan
    if (not np.isfinite(n)) or n < 4 or abs(r) >= 1:
        return np.nan
    z = np.arctanh(r) * np.sqrt(n - 3)
    return float(2 * (1 - norm.cdf(abs(z))))

print("Helpers ready: num(), _corr_ci(), _r_pvalue().")

In [ ]:
# === DO (6 of 24): E3, part 1 -- who counts as a retributivist, and on what ===
# THE HYPOTHESIS (prereg 2.4, E3)
#   "In their backward-looking explanations (the sentence justification
#   [Sentence_OpenEnd1] and the goal-ranking explanation [Q333]), retributivists use
#   more desert, proportionality and censure language than non-retributivists."
#
# WHO COUNTS AS A RETRIBUTIVIST HERE. Prioritization_1 == 1: the participant ranked
# "criminals deserve to be punished" FIRST. Prereg 5.3 calls this the primary
# categorical definition. It is a SURVEY-side variable, so the group assignment is
# entirely independent of anything the language models produced -- the contrast is
# text against an external classification, not text against itself.
#
# WHY Q333 AND Q1 ARE TESTED SEPARATELY AND NEVER POOLED. Both are backward-looking
# and most participants wrote both. Pooling would enter the same person twice as if
# they were two observations and would inflate every n in the table. Keeping them
# apart buys something better than a bigger n: the two items are near-replications
# of each other within the same study, so AGREEMENT OF THE CONTRAST ACROSS Q333 AND
# Q1 is the within-study replication. A facet effect that appears on one item and
# vanishes on the other is a finding about that item, not about retributivism.
#
# WORD COUNT. Prereg 6.7 makes word count a covariate across the family. The
# contrast implemented here is the unadjusted group difference, so fold n_words in
# as a covariate -- a regression on the group flag rather than a two-sample d --
# when the analysis-unit decision in MEASUREMENT_DESIGN 11.12 is ratified. That
# decision fixes the model form for E3 through E6 together.
E3_FACETS = list(GATE_BACKED_FACETS)
if not E3_FACETS:
    # An empty gate does not silence E3; it changes what E3 is. The fallback runs the
    # same contrast on the gate-eligible set, and this warning is the flag that those
    # rows are descriptive rather than gate-backed. Do not quote them as primary.
    print("WARNING: no facet cleared the gate. E3 runs on the gate-eligible set")
    print("         DESCRIPTIVELY. Every E3 row will say so in its gate_status.")
    E3_FACETS = list(GATE_ELIGIBLE)

def cohens_d(a, b):
    """Pooled-SD Cohen's d for two groups. NaN where it is undefined.

    Pooled rather than Glass's delta because neither group is a control:
    retributivists and everyone else are both substantive groups here.
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:          # or the pooled SD has no denominator
        return np.nan
    # ddof=1 on both variances; the pooled SD weights by degrees of freedom, which
    # matters because the retributivist group is much the smaller of the two.
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    # A zero pooled SD (a facet nobody's response triggered) yields no effect size
    # rather than an infinity.
    return float((a.mean() - b.mean()) / sp) if sp > 0 else np.nan

print(f"E3 facets: {', '.join(E3_FACETS)}  (items: Q333 and Q1, separately)")

In [ ]:
# === DO (7 of 24): E3, part 2 -- the contrast, one item at a time ===
e3_rows, E3_RAN = [], False
for it, tag in [("ranking", "Q333"), ("backward", "Q1")]:
    # The forward item is deliberately absent: E3 is about backward-looking
    # explanations, and the forward item is E6's business.
    # ONE WORKING NAME PER BLOCK: sub_it here, sub4 in E4, sub5 in E5. All three are
    # slices of the same table, and a name shared between two of them is how a cell
    # re-run out of order hands one block the other block's rows without saying so.
    sub_it = feat[feat["item_type"] == it].copy()
    if not SURVEY_OK or "Prioritization_1" not in sub_it.columns:
        print(f"  [{tag}] Prioritization_1 not available; E3 not run on this item.")
        continue
    # Coerced to a number first. Qualtrics exports the ranking as a label on some
    # settings and as a value on others, and a string "1" would compare False
    # against the integer 1 -- which would empty the retributivist group silently
    # rather than raise anything.
    grp_retrib = pd.to_numeric(sub_it["Prioritization_1"], errors="coerce") == 1
    for facet in E3_FACETS:
        col = f"ens_{facet}"
        if col not in sub_it.columns:
            continue
        # Retributivists and everyone else. Note that ~grp_retrib puts participants
        # with a MISSING Prioritization_1 in the comparison group, which is the
        # conservative direction: it can only dilute the contrast, never manufacture
        # one.
        d1 = sub_it.loc[grp_retrib, col].dropna()
        d0 = sub_it.loc[~grp_retrib, col].dropna()
        if len(d1) < 2 or len(d0) < 2:
            continue
        d_est = cohens_d(d1.values, d0.values)
        # The interval. Both groups are stacked into one vector with a group flag
        # beside it, and the bootstrap resamples that stack jointly, so the group
        # sizes vary from replicate to replicate exactly as they would in a fresh
        # sample. That is the right sampling model when group membership is observed
        # rather than assigned.
        vals = np.concatenate([d1.values, d0.values])
        flag = np.concatenate([np.ones(len(d1)), np.zeros(len(d0))])
        _, lo, hi = boot_ci(lambda v, g: cohens_d(v[g == 1], v[g == 0]), vals, flag)
        # Welch (equal_var=False) rather than Student: the groups are far apart in
        # size and there is no reason to assume the facet variance is the same in
        # both. This p goes on to the single BH-FDR family below.
        _t, p = ttest_ind(d1, d0, equal_var=False)
        e3_rows.append({"test": "E3", "item": tag, "facet": facet,
                        "n_retrib": len(d1), "n_other": len(d0),
                        "cohens_d": d_est, "d_lo": lo, "d_hi": hi,
                        "p": float(p), "family": FAMILY, "status": "computed",
                        "rests_on": rests(facet)})
        E3_RAN = True

e3_df = pd.DataFrame(e3_rows)

In [ ]:
# === DO (8 of 24): E3, part 3 -- read the table ===
# READING IT: a positive cohens_d means retributivists used MORE of that facet's
# language, which is the predicted direction for the retributive facets. A d whose
# interval spans zero is not an effect whatever its p, and the p column here is
# UNCORRECTED -- the BH-adjusted version is in the family table below, and that is
# the one to quote.
print("=== E3: retributivist vs other (Q333 and Q1 SEPARATELY, never pooled) ===")
if E3_RAN:
    print(e3_df.to_string(index=False,
          na_rep="--", float_format=lambda x: f"{x:.3f}"))
    print("\nAgreement of the facet contrast across Q333 and Q1 is the within-study")
    print("replication. A facet that shows the contrast on one item and not the")
    print("other is a finding about that item.")
else:
    # Said in words. An empty E3 block with no explanation would be read as a null.
    print("  NOT RUN. E3 needs Prioritization_1 from retribution_scored.csv, and it")
    print("  is not available in this run. That is a missing input, not a null.")
print("\nEXPLORATORY (prereg 2.4), inside the single E3-E7 BH-FDR family (6.7).")

In [ ]:
# === DO (9 of 24): E4, part 1 -- the study's central claim, and the threat to it ===
# THE HYPOTHESIS (prereg 2.4, E4)
#   "Within participants, principled-retributive language (desert, proportionality,
#   censure) appears together with vengeful language (revenge, infliction of
#   suffering) rather than in place of it. A positive correlation between the two
#   across responses would show that ordinary just deserts is bound up with
#   revenge." This is the claim the whole study is built around, which is why it
#   carries more scaffolding than any other test here.
#
# THE THREAT, IN THE REGISTRATION'S OWN WORDS. Prereg 6.7: "Because both sides come
# from the same automated scoring, one method's tendency to assign several facets at
# once could create this correlation on its own." A model asked in ONE call to rate
# desert and revenge on the same passage, under a rubric that teaches it these
# facets can co-occur, has every opportunity to produce the association from
# within-rater halo rather than from the text.
#
# THE THREE ANSWERS TO IT
#   (1) SEPARATE CALLS. Notebook 06 elicits desert and revenge in independent calls,
#       each blind to the other's answer. The split-call correlation is E4's PRIMARY
#       estimate. The joint-call number is retained and the difference between them
#       is reported as the halo diagnostic: if the association collapses when the
#       calls are separated, it was a rating artifact. Splitting the calls is a
#       registered deviation (prereg 6.7 says "Each model scores every response on
#       the facets" -- one pass, all facets) and it applies to the E4 pair only.
#   (2) THE PREREGISTERED WORD LIST. Prereg 6.7: "we require the same co-occurrence
#       to show up in the non-model word-list measure". Computed below, with its
#       interval, its joint-firing row count and its realized prevalence, so how
#       much power that check actually had is an observed number rather than an
#       assumption. It uses no model at all, which is what makes it independent.
#   (3) THE HUMAN GOLD SET, added alongside the word list and never instead of it
#       (design section 0). It runs only in the optional human arm.
#
# WHAT IS DELIBERATELY EXCLUDED. The survey r(deserved, revenge) of about .55 is a
# CONVERGENT REFERENCE that prereg 6.7 asks to be compared against, never a
# criterion: making it a criterion would install the survey as ground truth for a
# measure whose entire purpose is to reach what the survey cannot. The number is
# printed from the file below rather than taken from this comment. The holistic
# "both" rate is BARRED -- it is the same models' own summary judgment of the same
# passage, so it cannot corroborate their facet scores. It is printed as description.
E4_ITEM = "backward"      # the primary retributive site (prereg 5.15)
# sub4 is E4's own slice, numbered for its block. E3 above and E5 below hold their
# own slices of this same table under their own names, so re-running any one of the
# three cells cannot leave another block computing on a slice it did not take.
sub4 = feat[feat["item_type"] == E4_ITEM].copy()
print(f"E4 site: {E4_ITEM} (Sentence_OpenEnd1), {len(sub4):,} rows.")

In [ ]:
# === DO (9 of 24, cont.): E4's primary estimate, and the halo diagnostic ===
# THE CONTRACT. ens_desert_solo and ens_revenge_solo are the separately elicited
# scores. Without them E4 has no halo-resistant estimate, and the joint-call number
# must NOT quietly take its place: that is the one substitution this whole design is
# built to prevent. So the primary estimate is recorded as not run, loudly, and E4
# gives up its slot in the family rather than filling it with a number exposed to
# the artifact.
# HAS_SOLO was read off the pivoted table in DO 1 and is the one place this contract
# is decided; re-testing the columns here would be a second name for one contract, and
# two names for one contract eventually disagree.
E4_PRIMARY_OK = {"ens_desert_solo", "ens_revenge_solo"}.issubset(HAS_SOLO)
if E4_PRIMARY_OK:
    r_split, lo_s, hi_s, n_s = _corr_ci(sub4["ens_desert_solo"],
                                        sub4["ens_revenge_solo"])
    e4_primary_status = "computed"
else:
    r_split = lo_s = hi_s = np.nan; n_s = 0
    e4_primary_status = "not run"
    print("  " + "=" * 68)
    print("  E4'S PRIMARY ESTIMATE DID NOT RUN. The separate-call columns")
    print("  ens_desert_solo and ens_revenge_solo are not in ensemble_scores.csv.")
    print("  They come from notebook 06's split-call block, carried through")
    print("  notebook 07. Re-run 06 with that block enabled, then 07, then this.")
    print("  The joint-call number below is NOT a substitute: it is the estimate")
    print("  exposed to the halo artifact E4 exists to rule out, and E4 forfeits")
    print("  its slot in the corrected family rather than be represented by it.")
    print("  " + "=" * 68)

# The joint call: the preregistered one-pass scores. Reported in full, but tagged
# 'diagnostic' rather than with the family tag, so it cannot be mistaken for the
# estimate E4 rests on.
r_joint, lo_j, hi_j, n_j = _corr_ci(sub4.get("ens_desert"), sub4.get("ens_revenge"))
# halo > 0 means the joint call produced a stronger association than the split call
# did: part of the joint correlation was the rater seeing both facets in one glance
# rather than the text containing both.
halo = ((r_joint - r_split)
        if (not np.isnan(r_joint) and not np.isnan(r_split)) else np.nan)
print(f"  split-call (primary): r = {num(r_split)} [{num(lo_s)}, {num(hi_s)}], n = {n_s}")
print(f"  joint-call (diag):    r = {num(r_joint)} [{num(lo_j)}, {num(hi_j)}], n = {n_j}")
print(f"  halo (joint - split): {num(halo)}")

In [ ]:
# === DO (10 of 24): E4's PREREGISTERED non-model corroborator -- the word list ===
# Prereg 6.7 names this check and requires it: "we require the same co-occurrence to
# show up in the non-model word-list measure". It is computed and reported here
# whatever it returns, together with the realized prevalence and joint-firing count
# that determine how much power it had.
#
# dict_desert and dict_revenge are notebook 01's WORD-LIST hit counts, and the word
# list is what makes this leg independent: it uses no model at all. Reading ens_ here
# would turn the independent corroborator into the ensemble correlating with itself.
if {"dict_desert", "dict_revenge"}.issubset(sub4.columns):
    dd = sub4[["dict_desert", "dict_revenge"]].apply(pd.to_numeric,
                                                     errors="coerce").dropna()
    _d_pos, _r_pos = dd["dict_desert"] > 0, dd["dict_revenge"] > 0
    # Realized prevalence of each facet in the word list ON THIS CORPUS. Notebook
    # 01's sparsity table anticipates roughly 0.7% for revenge; these are observed.
    wl_desert_prev = float(_d_pos.mean()) if len(dd) else np.nan
    wl_revenge_prev = float(_r_pos.mean()) if len(dd) else np.nan
    # The count that actually determines the power of the preregistered check: how
    # many responses tripped BOTH word lists. If this is a handful, the check had
    # nothing to detect with.
    wl_joint_n = int((_d_pos & _r_pos).sum()) if len(dd) else 0
    r_dict, lo_d, hi_d, n_d = _corr_ci(dd["dict_desert"], dd["dict_revenge"])
else:
    r_dict = lo_d = hi_d = np.nan; n_d = 0
    wl_desert_prev = wl_revenge_prev = np.nan; wl_joint_n = 0

# WHETHER THIS LEG CARRIES EVIDENTIAL WEIGHT is decided by notebook 02's 5% coverage
# floor, and the verdict is CARRIED from that notebook rather than recomputed here.
# It is keyed on REVENGE because revenge is the sparser of the two and therefore the
# binding constraint on joint firing: desert can be common and the joint count still
# be near zero.
wl_leg_informative, wl_floor_note = "not run", "word_list_coverage.csv is not here"
if coverage is not None:
    _row = coverage[(coverage["facet"] == "revenge") &
                    (coverage["scope"] == E4_ITEM)]
    if len(_row):
        _clears = str(_row["clears_floor"].iloc[0]).strip().lower()
        wl_leg_informative = {
            "yes": "informative -- the word list clears the 5% floor here",
            "no": "below floor -- uninformative by base rate, NOT disconfirmation",
        }.get(_clears, "not run")
        wl_floor_note = f"notebook 02 coverage verdict for revenge at {E4_ITEM}"
    else:
        wl_floor_note = f"no coverage row for revenge at {E4_ITEM}"

# WHAT THE EXPORTED status COLUMN IS ALLOWED TO SAY. It says whether the number was
# COMPUTED. It does not say whether the number is INFORMATIVE -- that is the coverage
# verdict, and it travels beside it in the note column. Putting the verdict in status
# would put this row on the record as "not run" whenever word_list_coverage.csv
# happened to be absent, and prereg 6.7 requires this corroborator to have been RUN:
# anyone filtering the export on status == "computed" would drop the one row the
# registration insists on.
if not {"dict_desert", "dict_revenge"}.issubset(sub4.columns):
    wl_status = "not run -- no dict_desert/dict_revenge columns in the merged table"
elif n_d == 0:
    wl_status = "not run -- the word-list columns are present but hold no values"
else:
    wl_status = "computed"

In [ ]:
# === DO (10 of 24, cont.): read the preregistered check, with its power ===
# THE POINT OF PRINTING THE PREVALENCE BESIDE THE RESULT. A correlation computed on
# a measure that fires on well under one response in a hundred is pinned near zero
# by base rate: it can neither corroborate E4 nor refute it. Reported alone, a near
# zero here invites a reader to treat a near-empty measure as a disconfirming
# finding. Reported beside its prevalence and its joint-firing count, it is what it
# actually is -- an observed limitation of this corpus.
print("PREREG 6.7 word-list corroborator for E4 (computed, not skipped):")
print(f"  r(dict_desert, dict_revenge) = {num(r_dict)} "
      f"[{num(lo_d)}, {num(hi_d)}], n = {n_d}")
print(f"  word-list desert prevalence  = "
      + ("--" if np.isnan(wl_desert_prev) else f"{wl_desert_prev:.2%}"))
print(f"  word-list revenge prevalence = "
      + ("--" if np.isnan(wl_revenge_prev) else f"{wl_revenge_prev:.2%}")
      + "   <- the binding constraint")
print(f"  rows firing BOTH lists       = {wl_joint_n}")
print(f"  5% coverage floor verdict    = {wl_leg_informative}  ({wl_floor_note})")
print("  -> If this leg is below the floor, report it in the RESULTS as")
print("     UNINFORMATIVE BY BASE RATE, an observed limitation with its counts.")
print("     It is NOT disconfirmation of E4, and a joint-firing count in single")
print("     digits is not a null result about co-occurrence.")

In [ ]:
# === DO (11 of 24): E4's added human corroborator, and the survey reference ===
# THE HUMAN GOLD SET is a second, independent non-model corroborator -- an ADDITION
# to the preregistered word-list check, never a substitute for it (design section 0).
# Prereg 6.7 has the model ensemble code the corpus IN PLACE OF human raters and
# prescribes no human-coding step, so this arm is a DEVIATION and it is switched off
# by default. Where it did not run this says so in words. It never says nothing, and
# it never reports a blank that could be read as a failed check.
r_gold = lo_g = hi_g = np.nan; n_g = 0; coassign_rate = np.nan
gold_status = "not run"
gold_note = ("HUMAN_GOLD is False: prereg 6.7 prescribes no human-coding step, so "
             "no human has read these responses anywhere in this chain")
if gold is not None and {"desert", "revenge", "item_type"}.issubset(gold.columns):
    # The gold sample was drawn on survey signal and a random slice only -- never on
    # ensemble or embedding output -- so the coders could not have been steered
    # toward the ensemble's reading of any response. This uses the WHOLE gold set for
    # the item rather than the TEST half: it is a corroborating estimate of the
    # association, not a validation of the ensemble, so the SEED/TEST discipline that
    # governs the gate does not bind here.
    gm = gold[gold["item_type"] == E4_ITEM][["desert", "revenge"]].apply(
        pd.to_numeric, errors="coerce").dropna()
    r_gold, lo_g, hi_g, n_g = _corr_ci(gm["desert"], gm["revenge"])
    # The plainer statement of the same thing: what share of coded responses did
    # humans mark for BOTH desert and revenge? A correlation can be produced by
    # co-ABSENCE; a co-assignment rate cannot.
    coassign_rate = float(((gm["desert"] > 0) & (gm["revenge"] > 0)).mean()) if len(gm) else np.nan
    gold_status, gold_note = "computed", "human coders, blind to the ensemble"

# THE SURVEY CONVERGENT REFERENCE. Prereg 6.7 asks for this comparison explicitly:
# "we compare it with the relationship between the desert and revenge scales in the
# survey". Participant-level, so it is computed on `surv` directly and not on the
# item table, or every person would count three times. Reference only -- it sets no
# threshold for E4, because r = .55 on N = 948 is high precisely BECAUSE the survey's desert and
# revenge constructs are not cleanly separable.
r_survey, n_survey = np.nan, 0
if SURVEY_OK and {"deserved", "revenge"}.issubset(surv.columns):
    sv = surv[["deserved", "revenge"]].apply(pd.to_numeric, errors="coerce").dropna()
    r_survey = float(pearsonr(sv["deserved"], sv["revenge"])[0]) if len(sv) > 2 else np.nan
    n_survey = len(sv)

# THE SAME REFERENCE ON THE LITERAL REVENGE ITEMS (design section 0, the
# revenge-scale row). revenge_4 in the registered scale asks for harsher punishment
# rather than for getting even, so the desert-revenge survey correlation is reported
# on revenge_1 and _2 as well, scored with the at-least-half rule. Reference only,
# exactly like the registered one; it sets no threshold for E4 either.
r_survey_lit, n_survey_lit = np.nan, 0
if SURVEY_OK and {"deserved", "revenge_1", "revenge_2"}.issubset(surv.columns):
    _lit = surv[["revenge_1", "revenge_2"]].apply(pd.to_numeric, errors="coerce")
    _svl = pd.DataFrame({
        "deserved": pd.to_numeric(surv["deserved"], errors="coerce"),
        "revenge_lit": _lit.mean(axis=1).where(_lit.notna().sum(axis=1) >= 1)}).dropna()
    r_survey_lit = (float(pearsonr(_svl["deserved"], _svl["revenge_lit"])[0])
                    if len(_svl) > 2 else np.nan)
    n_survey_lit = len(_svl)

In [ ]:
# === DO (11 of 24, cont.): the two quantities that are barred from E4's evidence ===
# THE HOLISTIC "BOTH" RATE. Share of responses the ensemble's majority holistic call
# labelled 'both'. It looks like corroboration and it is not: it is the same models
# summarizing the same passage in the same pass, so it can only reproduce whatever
# produced the facet scores, halo included. Printed as description, barred from the
# evidence set (design section 9, rule 6).
both_rate = (float((sub4["holistic"].astype(str).str.lower() == "both").mean())
             if "holistic" in sub4.columns else np.nan)

# THE EMBEDDING STREAM, EXCLUDED BY DESIGN. Notebook 04 builds each facet's reference
# pool from DISJOINT seed exemplars, so two facets are pushed apart by construction
# and joint firing is depressed mechanically. A co-occurrence estimate taken from
# this stream would be measuring the instrument rather than the responses. It is
# computed anyway, and only so that the exclusion is a visible number rather than an
# assertion -- it corroborates nothing and takes no slot in the family.
# The SEED rows are dropped first: the pools were built partly from seed-half
# responses, so a similarity computed on a SEED row is partly a check of a stream
# against the very responses that seeded it.
emb_r, emb_n, emb_by = np.nan, 0, "not run"
if sim is not None:
    # facet_similarity.csv is keyed on (response_id, item_type, facet) as well, and
    # the pivot below would average a duplicated row into one cell. This input is
    # OPTIONAL and the figure it produces is excluded from E4's evidence by design, so
    # a duplicate is reported rather than raised: it costs the exclusion figure some
    # precision and costs no analysis anything.
    _dup_sim = int(sim.duplicated(["response_id", "item_type", "facet"]).sum())
    if _dup_sim:
        print(f"  note: facet_similarity.csv holds {_dup_sim} duplicated "
              "(response_id, item_type, facet) row(s), which the excluded embedding "
              "figure below averages. No analysis in this notebook uses it.")
    _s = sim[sim["item_type"] == E4_ITEM]
    if "gold_split" in _s.columns:
        _s = _s[_s["gold_split"] != "SEED"]
    _vals = []
    for c in [c for c in _s.columns if c.endswith("_sim")]:
        w = _s[_s["facet"].isin(["desert", "revenge"])].pivot_table(
            index="response_id", columns="facet", values=c)
        if {"desert", "revenge"}.issubset(w.columns):
            w = w.dropna()
            if len(w) > 2 and w["desert"].std() > 0 and w["revenge"].std() > 0:
                _vals.append((c[:-4], float(pearsonr(w["desert"], w["revenge"])[0]),
                              len(w)))
    if _vals:
        emb_r = float(np.median([v[1] for v in _vals]))
        emb_n = max(v[2] for v in _vals)
        emb_by = "|".join(f"{k}={r:.3f}" for k, r, _ in _vals)
print(f"  holistic 'both' rate (descriptive, barred): "
      + ("--" if np.isnan(both_rate) else f"{both_rate:.1%}"))
print(f"  embedding desert x revenge (EXCLUDED by design): {num(emb_r)} "
      f"(n = {emb_n}; by encoder: {emb_by})")

In [ ]:
# === DO (12 of 24): E4, one table, every quantity, each with its standing ===
# The `family` column is what fixes each row's status, and only the SPLIT_CALL row
# carries the family tag. That tag is what the correction cell selects on, which is
# why the corroborators, the reference and the diagnostics do not each consume an
# FDR slot -- charging them a slot would penalize the analysis for having checked
# its own work.
e4_df = pd.DataFrame([
    {"test": "E4", "estimate": "desert_x_revenge_SPLIT_CALL", "r": r_split,
     "lo": lo_s, "hi": hi_s, "n": n_s, "family": FAMILY,
     "status": e4_primary_status, "rests_on": rests("desert", "revenge")},
    {"test": "E4", "estimate": "desert_x_revenge_JOINT_CALL_halo_diag", "r": r_joint,
     "lo": lo_j, "hi": hi_j, "n": n_j, "family": "diagnostic",
     "status": "computed", "rests_on": rests("desert", "revenge")},
    {"test": "E4", "estimate": "HALO(joint-split)", "r": halo, "lo": np.nan,
     "hi": np.nan, "n": np.nan, "family": "diagnostic",
     "status": e4_primary_status, "rests_on": rests("desert", "revenge")},
    {"test": "E4", "estimate": "WORDLIST_desert_x_revenge(PREREG_corroborator_6.7)",
     "r": r_dict, "lo": lo_d, "hi": hi_d, "n": n_d,
     "family": "prereg_corroborator_wordlist", "status": wl_status,
     "rests_on": rests("desert", "revenge")},
    {"test": "E4", "estimate": "wordlist_revenge_prevalence(power_of_prereg_check)",
     "r": wl_revenge_prev, "lo": np.nan, "hi": np.nan, "n": n_d,
     "family": "prereg_corroborator_wordlist", "status": "computed",
     "rests_on": rests("revenge")},
    {"test": "E4", "estimate": "wordlist_desert_prevalence", "r": wl_desert_prev,
     "lo": np.nan, "hi": np.nan, "n": n_d,
     "family": "prereg_corroborator_wordlist", "status": "computed",
     "rests_on": rests("desert")},
    {"test": "E4", "estimate": "wordlist_JOINT_desert&revenge_row_count",
     "r": float(wl_joint_n), "lo": np.nan, "hi": np.nan, "n": n_d,
     "family": "prereg_corroborator_wordlist", "status": "computed",
     "rests_on": rests("desert", "revenge")},
    {"test": "E4", "estimate": "HUMAN_GOLD_desert_x_revenge(ADDED_corroborator)",
     "r": r_gold, "lo": lo_g, "hi": hi_g, "n": n_g,
     "family": "added_corroborator_human", "status": gold_status,
     "rests_on": rests("desert", "revenge")},
    {"test": "E4", "estimate": "human_gold_coassign_rate", "r": coassign_rate,
     "lo": np.nan, "hi": np.nan, "n": n_g, "family": "added_corroborator_human",
     "status": gold_status, "rests_on": rests("desert", "revenge")},
    {"test": "E4", "estimate": "SURVEY_deserved_x_revenge(PREREG_comparison_6.7)",
     "r": r_survey, "lo": np.nan, "hi": np.nan, "n": n_survey,
     "family": "prereg_convergent_reference",
     "status": "computed" if SURVEY_OK else "not run", "rests_on": ""},
    {"test": "E4", "estimate": "SURVEY_deserved_x_revenge_LITERAL_items(added_reference)",
     "r": r_survey_lit, "lo": np.nan, "hi": np.nan, "n": n_survey_lit,
     "family": "prereg_convergent_reference",
     "status": "computed" if (SURVEY_OK and n_survey_lit) else "not run",
     "rests_on": ""},
    {"test": "E4", "estimate": "holistic_both_rate(descriptive_barred)",
     "r": both_rate, "lo": np.nan, "hi": np.nan, "n": np.nan,
     "family": "descriptive_barred", "status": "computed", "rests_on": ""},
    {"test": "E4", "estimate": "EMBEDDING_desert_x_revenge(EXCLUDED_by_design)",
     "r": emb_r, "lo": np.nan, "hi": np.nan, "n": emb_n,
     "family": "excluded_by_design_embedding", "status": "computed",
     "rests_on": rests("desert", "revenge")},
])
# The note column carries what status is not allowed to carry: why a row did not
# run, and -- for the preregistered word-list row -- whether the measure fired often
# enough to be informative. Both rows were computed either way.
e4_df["note"] = ""
e4_df.loc[e4_df["family"] == "added_corroborator_human", "note"] = gold_note
e4_df.loc[e4_df["estimate"].str.startswith("WORDLIST_"), "note"] = (
    f"5% coverage floor: {wl_leg_informative} ({wl_floor_note})")

In [ ]:
# === DO (13 of 24): E4, read the output ===
# READING IT
#   SPLIT_CALL positive with an interval clear of zero = E4's claim, on the estimate
#     that carries the weight.
#   HALO near zero = the joint and split calls agree, so the association is not a
#     rating artifact.
#   A large positive HALO beside a small SPLIT_CALL = the joint-call number was
#     mostly halo, and the preregistered one-pass estimate should be read that way.
#   A word-list row count in single digits = the preregistered check had no power
#     here. Say so with the counts; it is not evidence against E4.
print("=== E4: within-person desert x revenge co-occurrence ===")
print(e4_df[["estimate", "r", "lo", "hi", "n", "family", "status"]].to_string(
    index=False, na_rep="--", float_format=lambda x: f"{x:.3f}"))
print(f"\nHalo diagnostic: joint({num(r_joint)}) - split({num(r_split)}) = "
      f"{num(halo)}. A collapse under split scoring means rating artifact.")
print("Primary weight sits on the SPLIT-CALL estimate. Corroboration is the PREREG")
print("word list (above, with its prevalence) PLUS the added human gold set, which")
print(f"in this run is: {gold_status}.")
print("The survey r is a convergent reference and sets no threshold; the holistic")
print("'both' rate and the embedding stream are barred and excluded respectively.")
print("E4 remains EXPLORATORY (prereg 2.4) and inside the one BH-FDR family (6.7).")

In [ ]:
# === DO (14 of 24): E5 / text-side H6 -- do desert and proportionality hold together? ===
# THE HYPOTHESIS (prereg 2.4, E5)
#   "Desert language and proportionality language do not hold together in the
#   open-ended responses, the text version of H6." H6 itself (prereg 6.4) asks
#   whether just deserts coheres on the closed-ended side; E5 asks the same question
#   of the text.
#
# TWO SIDES, TWO STANDINGS. The SURVEY side stands on its own as H6's evidence and
# is not part of this exploratory family; it is computed here (r is about .085
# between `deserved` and `proportionality_jd`, printed from the file below rather
# than taken from this comment) so the two sides can be read against each other in
# one place. The TEXT side is the E5 test proper: it is EXPLORATORY
# and it is INSIDE the single E3-E7 BH-FDR family. Note the distinction that governs
# this cell -- the FACET is descriptive-only (proportionality's demotion is a
# registered deviation; prereg 5.15 does anchor it), but the TEST that facet
# produces is corrected along with the rest of the family. Holding it out would
# understate the multiplicity the analysis actually incurs.
#
# WHY ONLY TWO OF THE FOUR STREAMS COUNT. The co-occurrence estimate uses ONLY the
# separately-called ensemble and the human gold set. The word list and the embedding
# stream are EXCLUDED, and the reason is mechanical rather than evidential:
#   - notebook 01's refined word list enforces TOKEN DISJOINTNESS, so no word counts
#     toward two opposing facets and two facets cannot easily fire on one short
#     response;
#   - notebook 04's reference pools are built from DISJOINT seed exemplars per
#     facet, so the similarity scores are pushed apart by construction.
# Both instruments depress joint firing by design. A near-zero co-occurrence coming
# out of them would be measuring the instrument, not the responses -- which is
# exactly the inference E5 must not make, since E5's own hypothesis is that the
# co-occurrence is absent.
H6_ITEM = "backward"

# The survey side, participant-level so nobody is counted three times.
r_survey_h6, n_survey_h6 = np.nan, 0
if SURVEY_OK and {"deserved", "proportionality_jd"}.issubset(surv.columns):
    sv = surv[["deserved", "proportionality_jd"]].apply(pd.to_numeric,
                                                        errors="coerce").dropna()
    r_survey_h6 = (float(pearsonr(sv["deserved"], sv["proportionality_jd"])[0])
                   if len(sv) > 2 else np.nan)
    n_survey_h6 = len(sv)

In [ ]:
# === DO (14 of 24, cont.): the text side, which is the E5 test proper ===
# Desert uses its split-call score where notebook 06 provided one and falls back to
# the joint-call score otherwise. Only desert and revenge were re-elicited in
# separate calls, because E4's pair is the one exposed to halo; proportionality has
# only the joint-call score by construction.
sub5 = feat[feat["item_type"] == H6_ITEM].copy()
desert_col = "ens_desert_solo" if "ens_desert_solo" in sub5.columns else "ens_desert"
r_ens_h6 = p_ens_h6 = np.nan; n_ens_h6 = 0; e5_status = "not run"
if {desert_col, "ens_proportionality"}.issubset(sub5.columns):
    te = sub5[[desert_col, "ens_proportionality"]].apply(pd.to_numeric,
                                                         errors="coerce").dropna()
    n_ens_h6 = len(te)
    # The zero-variance branch is the live one here: at roughly a 1% proportionality
    # base rate the ensemble column can be effectively constant, and a constant has
    # no correlation. That outcome is the null-by-absence case, not a finding.
    if (len(te) > 2 and te["ens_proportionality"].std() > 0
            and te[desert_col].std() > 0):
        _r, _p = pearsonr(te[desert_col], te["ens_proportionality"])
        r_ens_h6, p_ens_h6, e5_status = float(_r), float(_p), "computed"
    else:
        e5_status = "not run -- no variance in ens_proportionality on this item"
print(f"E5 text side uses {desert_col} x ens_proportionality at the {H6_ITEM} item.")
print(f"  r = {num(r_ens_h6)}, p(uncorrected) = {num(p_ens_h6, 5)}, n = {n_ens_h6}")

In [ ]:
# === DO (15 of 24): E5's verification -- could the co-occurrence have appeared? ===
# THE NULL-BY-ABSENCE RULE. Before any absence can be read as a dissociation, it has
# to have been POSSIBLE to mark desert and proportionality on the same response. If
# proportionality language is essentially not in the corpus, there were too few
# proportionality-positive cases for a dissociation to appear in, and the result is
# a DESCRIPTIVE NULL-BY-ABSENCE rather than a null.
#
# The operative verdict is CARRIED from notebook 10, which computed the realized
# proportionality base rate against a 10% floor. Recomputing it here would put the
# same rule in two places, where the two copies eventually disagree.
#
# The design states this check on the HUMAN base rate -- whether coders CAN and
# sometimes DO co-assign desert and proportionality -- because the humans are the
# better witness to whether the construct was present at all. Where the human arm
# runs, that is the verification which licenses the inference, and it comes from
# coders who never saw the ensemble's output: if humans co-assign at a reasonable
# rate and the ensemble shows no association, that is a finding about the text; if
# the humans almost never marked proportionality at all, there was nothing to
# co-assign and no finding is available either way. That arm is off by
# default (prereg 6.7 prescribes no human-coding step), so the human quantities read
# 'not run' and the ensemble-based verdict from notebook 10 is what E5 is reported
# against. Say which of the two you are quoting.
coassign_h6 = prop_gold_rate = np.nan
h6_gold_status = "not run"
if gold is not None and {"desert", "proportionality", "item_type"}.issubset(gold.columns):
    gm6 = gold[gold["item_type"] == H6_ITEM][["desert", "proportionality"]].apply(
        pd.to_numeric, errors="coerce").dropna()
    if len(gm6):
        coassign_h6 = float(((gm6["desert"] > 0) & (gm6["proportionality"] > 0)).mean())
        prop_gold_rate = float((gm6["proportionality"] > 0).mean())
        h6_gold_status = "computed"

if PROP_NULL_BY_ABSENCE is None:
    null_by_absence = "not run"
    nba_note = "gate_results.csv carries no prop_null_by_absence column"
else:
    null_by_absence = "yes" if PROP_NULL_BY_ABSENCE else "no"
    nba_note = "carried from notebook 10: realized ensemble base rate vs the 10% floor"
print(f"Descriptive null-by-absence: {null_by_absence}  ({nba_note})")
print(f"Human gold verification: {h6_gold_status}"
      + ("" if h6_gold_status == "computed" else
         " -- no human has co-assigned anything in this chain"))

In [ ]:
# === DO (15 of 24, cont.): E5's table, and how to read it ===
# Note the `family` tags: only the ENSEMBLE text-side row carries the family tag and
# takes an FDR slot. The survey row is standing H6 evidence outside this family, and
# the two gold rows are verification quantities rather than hypothesis tests.
e5_df = pd.DataFrame([
    {"test": "E5_H6", "estimate": "SURVEY_deserved_x_proportionality(standing_evidence)",
     "value": r_survey_h6, "p": np.nan, "n": n_survey_h6, "family": "survey_evidence",
     "status": "computed" if SURVEY_OK else "not run", "rests_on": ""},
    {"test": "E5_H6", "estimate": "ENSEMBLE_desert_x_proportionality(text_side)",
     "value": r_ens_h6, "p": p_ens_h6, "n": n_ens_h6, "family": FAMILY,
     "status": e5_status, "rests_on": rests("desert", "proportionality")},
    {"test": "E5_H6", "estimate": "human_gold_desert&proportionality_coassign_rate",
     "value": coassign_h6, "p": np.nan, "n": np.nan, "family": "verification",
     "status": h6_gold_status, "rests_on": rests("desert", "proportionality")},
    {"test": "E5_H6", "estimate": "human_gold_proportionality_base_rate",
     "value": prop_gold_rate, "p": np.nan, "n": np.nan, "family": "verification",
     "status": h6_gold_status, "rests_on": rests("proportionality")},
    {"test": "E5_H6", "estimate": "descriptive_null_by_absence(carried_from_nb10)",
     "value": np.nan, "p": np.nan, "n": np.nan, "family": "verification",
     "status": "computed" if PROP_NULL_BY_ABSENCE is not None else "not run",
     "rests_on": rests("proportionality")},
])
# status says whether a quantity was computed, in the same words everywhere in this
# file. The verdict itself -- yes or no -- is a value, and it goes in the note column
# beside the row, because a status column that reads "yes" on one row and "not run"
# on the next cannot be filtered or counted.
#
# The text-side estimate keeps its name whichever desert column produced it, because
# the correction cell selects this row BY NAME and a name that moved with the data
# would let the family lose its E5 slot silently. Which column it was is therefore
# recorded here instead: ens_desert_solo is the separately elicited score and
# ens_desert the joint-call one, and E4's whole design turns on not confusing them.
e5_df["note"] = ""
e5_df.loc[e5_df["estimate"] == "ENSEMBLE_desert_x_proportionality(text_side)",
          "note"] = f"desert column used: {desert_col}"
e5_df.loc[e5_df["estimate"].str.startswith("descriptive_null_by_absence"),
          "note"] = f"verdict: {null_by_absence} -- {nba_note}"
# READING IT: a near-zero text-side r with null_by_absence = yes means the corpus
# contained too little proportionality language to test the dissociation. Report the
# base rate, not the r. A near-zero r with null_by_absence = no, and a co-assignment
# rate comfortably above zero, is the case in which E5's predicted dissociation
# actually has support.
print("=== E5 / text-side H6 (desert vs proportionality) ===")
print(e5_df.to_string(index=False,
      na_rep="--", float_format=lambda x: f"{x:.3f}"))
print("\nThe word-list and embedding streams are EXCLUDED from the text-side")
print("estimate: both enforce disjointness and so depress joint firing by design.")
print("EXPLORATORY (prereg 2.4) and INSIDE the single E3-E7 BH-FDR family (6.7).")
print("The FACET is descriptive-only, a registered deviation. The TEST is still")
print("corrected with the rest of the family.")

In [ ]:
# === DO (16 of 24): E6 -- the backward-minus-forward register shift ===
# THE HYPOTHESIS (prereg 2.4, E6)
#   "Backward-looking explanations (why this sentence) are more retributive, and
#   forward-looking explanations (what the sentence should accomplish) more
#   consequentialist; the difference between them within a person indexes a
#   retributive versus consequentialist motive."
#   Prereg 5.15 is why the two items were never combined in the first place: the
#   justification looks back at what the offender did, the accomplishment item looks
#   ahead, and merging them would mix the two kinds of reasoning that this test
#   measures the distance between.
#
# WHY WITHIN-PERSON. delta = facet(backward) - facet(forward), computed for
# participants who wrote both. Everything that varies between people -- verbosity,
# vocabulary, vignette, how punitive they are -- is differenced out, so what remains
# is the shift in register inside one person's own writing. That is a far stronger
# design than comparing two groups of responses.
#
# E6 IS INSIDE THE FAMILY. Prereg 6.7: "E3 through E7 form one exploratory family;
# we control the false discovery rate within it." E6 is not exempt. This block tests
# ALL NINE facets -- including the descriptive-only ones and the three
# consequentialist goals -- because the register shift is a claim about every one of
# them, and that is a large part of why the family is the size it is.
bw = feat[feat["item_type"] == "backward"].set_index("response_id")
fw = feat[feat["item_type"] == "forward"].set_index("response_id")
# The paired set: participants who wrote a substantive answer to BOTH items.
common = bw.index.intersection(fw.index)
# THE HAZARD THIS CELL CARRIES, AND WHERE IT IS ACTUALLY CLOSED. The alignment above
# assumes exactly one backward row and one forward row per participant. A duplicated
# row makes the two .loc calls return more values than `common` holds and misaligns
# the subtraction silently: every facet's n below rises while the paired count printed
# on the next line goes on reporting the true number of people. The contract is
# enforced IN THIS NOTEBOOK, in two places -- the LOAD cell asserts ensemble_scores.csv
# carries no duplicated (response_id, item_type, facet) row, and DO 2 asserts the
# word-list frame is unique on (response_id, item_type) before merging it in and that
# the merge did not change the row count. Nothing upstream checks it: notebook 00
# checks openends_long.csv and no notebook after it re-checks.
print(f"E6 paired participants (wrote both items): {len(common):,}")

In [ ]:
# === DO (16 of 24, cont.): the nine differences ===
e6_rows = []
for facet in FACETS:
    col = f"ens_{facet}"
    if col not in bw.columns or col not in fw.columns:
        continue
    # dropna AFTER subtracting, so a participant missing either side drops out of
    # that facet's pair set rather than contributing half a difference.
    delta = (bw.loc[common, col] - fw.loc[common, col]).dropna()
    if len(delta) < 3:
        continue
    # The interval on the mean shift. The interval, not the p, is what the effect
    # should be read from.
    est, lo, hi = boot_ci(lambda a: np.nanmean(a), delta.values)
    # One-sample test that the within-person register shift differs from zero; this
    # p feeds the family. Guarded on nonzero spread: a facet that fires identically
    # on both items (commonly, never on either) has no variance to test and records
    # a NaN p, which the correction cell then drops.
    if float(np.nanstd(delta.values)) > 0:
        _t, _p = ttest_1samp(delta.values, 0.0, nan_policy="omit")
        _p = float(_p)
    else:
        _p = np.nan
    # The predicted direction depends on this label: retributive facets positive,
    # consequentialist negative. The class is about the CONSTRUCT, not about the
    # facet's gate standing. BOTH lists are consulted, never one with the other
    # assumed: the setup cell has already refused to run if any facet is missing from
    # both, and reading only RETRIBUTIVE here would hand a tenth facet the
    # consequentialist prediction by default and invert its expected sign.
    klass = ("retributive" if facet in RETRIBUTIVE else
             "consequentialist" if facet in CONSEQUENTIALIST else "UNCLASSIFIED")
    e6_rows.append({"test": "E6", "facet": facet, "class": klass, "n": len(delta),
                    "mean_delta_bw_minus_fw": est, "lo": lo, "hi": hi, "p": _p,
                    "family": FAMILY,
                    "status": "computed" if not np.isnan(_p) else
                              "not run -- no variance in the difference",
                    "rests_on": rests(facet)})

e6_df = pd.DataFrame(e6_rows)

In [ ]:
# === DO (17 of 24): E6, read the table ===
# READING IT: the prediction is a SIGN PATTERN, not a set of individual
# significances -- positive deltas down the retributive rows and negative deltas
# down the consequentialist rows. A single facet crossing zero matters less than the
# pattern breaking. Descriptive-only facets appear here, and their tests are
# corrected with the rest; their gate_status column says what they are.
print("=== E6: backward minus forward (positive => the facet peaks on backward) ===")
if len(e6_df):
    print(e6_df.to_string(index=False,
          na_rep="--", float_format=lambda x: f"{x:.3f}"))
    _ret = e6_df[e6_df["class"] == "retributive"]["mean_delta_bw_minus_fw"]
    _con = e6_df[e6_df["class"] == "consequentialist"]["mean_delta_bw_minus_fw"]
    print(f"\n  retributive facets positive:      "
          f"{int((_ret > 0).sum())} of {len(_ret)}")
    print(f"  consequentialist goals negative:  "
          f"{int((_con < 0).sum())} of {len(_con)}")
    print("  That pair of counts IS the E6 result. Read it before any single row.")
else:
    print("  NOT RUN: no facet had enough paired backward/forward rows.")
print("\nEXPLORATORY (prereg 2.4); INSIDE the single E3-E7 BH-FDR family (6.7).")

In [ ]:
# === DO (18 of 24): E7 -- does facet language predict the sentence beyond self-report? ===
# THE HYPOTHESIS (prereg 2.4, E7)
#   "The language facets agree with the survey retributivism measures and add
#   prediction of the sentence beyond the self-report scales." Implemented as a
#   two-step hierarchical regression: step 1 is the survey scale alone, step 2 adds
#   the gate-backed facet scores and word count, and the quantity of interest is the
#   increment.
#
# ############################################################################
# # THE ONE MISTAKE THAT WOULD INVALIDATE E7                                  #
# # Predict Sentence1_z from `punitiveness`, the EIGHT-ITEM self-report scale. #
# # NEVER from `punitiveness_full`. The full index INCLUDES the sentence       #
# # itself as a component: it correlates r = .86 with Sentence1_z, against     #
# # r = .47 for the 8-item scale. Regressing the sentence on an index that     #
# # embeds the sentence puts the outcome on both sides of the equation,        #
# # inflates step 1's R2 toward the ceiling, and leaves the facets with almost #
# # no residual variance to explain -- so E7 would report a null that is pure  #
# # construction. Prereg 6.7 sets exactly this restriction: "using the         #
# # eight-item punitiveness scale rather than the full index, because the full #
# # index already contains the sentence." A one-word edit here silently        #
# # destroys the test without raising anything.                                #
# ############################################################################
#
# WHY NEITHER MODEL CARRIES A VIGNETTE TERM. Sentence1_z arrives already standardized
# WITHIN VIGNETTE from the R pipeline (prereg 6.1), so the three randomly assigned
# vignettes' different severity levels have already been removed from the outcome.
# Adding a vignette term here would adjust for the same thing a second time.
#
# E7 DOES NOT DEGRADE GRACEFULLY. Where a required column is missing it is recorded
# as NOT RUN and the wrong column is never substituted -- substitution is the exact
# failure this block is built to prevent, and it would be invisible in the output.
E7_OK = SURVEY_OK
_e7_missing = [c for c in ["punitiveness", "Sentence1_z"]
               if not (SURVEY_OK and c in surv.columns)]
if _e7_missing:
    E7_OK = False
    print("  E7 NOT RUN: " + ", ".join(_e7_missing) + " not available.")
    print("  punitiveness_full is NEVER substituted for the 8-item scale, so E7 is")
    print("  reported as not run rather than as a null.")
# The full index may legitimately exist in the merged table. The guard prints so a
# reader of the output can see the column was present and deliberately unused.
if "punitiveness_full" in feat.columns:
    print("  GUARD: punitiveness_full is present in the merge and is NEVER used.")

In [ ]:
# === DO (19 of 24): E7's two nested models ===
# The site is the backward item -- Sentence_OpenEnd1, the justification for the very
# sentence being predicted, which is the register where desert language lives
# (prereg 5.15). One row per participant there.
e7_site = feat[feat["item_type"] == "backward"].copy()
# Gate-backed facets carry the primary version of this test; the fallback keeps E7
# runnable on the gate-eligible set, descriptively, when the gate returns empty.
E7_FACETS = list(GATE_BACKED_FACETS) or list(GATE_ELIGIBLE)
if not GATE_BACKED_FACETS:
    # Said out loud, for the reason E3 says it: without this line an all-descriptive
    # run and a gate-backed run produce identical E7 output on screen -- same R2, same
    # F, same p -- and the standing is recoverable only from the exported column.
    print("WARNING: no facet cleared the gate. E7 runs on the gate-eligible set")
    print("         DESCRIPTIVELY. Its row says so in gate_status; do not quote the")
    print("         increment as a gate-backed result.")
base = full = None
delta_r2 = f_stat = p_inc = np.nan; n_e7 = 0; k1 = k2 = 0
if E7_OK:
    model_cols = ["Sentence1_z", "punitiveness"] + [f"ens_{f}" for f in E7_FACETS]
    if "n_words" in e7_site.columns:      # word count as a covariate, prereg 6.7
        model_cols.append("n_words")
    # Listwise deletion -- dropping any participant missing ANY column either model
    # uses -- applied ONCE, before either model is fitted. This is what makes the two
    # models nested: step 1 and step 2 must be estimated on identical rows, or the
    # difference between their R2 values is not an increment at all.
    d = e7_site[[c for c in model_cols if c in e7_site.columns]].apply(
        pd.to_numeric, errors="coerce").dropna()
    n_e7 = len(d)
    _have = [f for f in E7_FACETS if f"ens_{f}" in d.columns]
    if n_e7 > (len(_have) + 4) and d["punitiveness"].std() > 0:
        base = smf.ols("Sentence1_z ~ punitiveness", data=d).fit()   # 8-item, NOT _full
        rhs = "punitiveness" + "".join(f" + ens_{f}" for f in _have)
        if "n_words" in d.columns:
            rhs += " + n_words"
        full = smf.ols(f"Sentence1_z ~ {rhs}", data=d).fit()
        # The headline quantity: how much of the sentence's variance the LANGUAGE
        # explains that the SELF-REPORT did not. That is the whole point of adding a
        # text measure to this study.
        delta_r2 = full.rsquared - base.rsquared
        # k2 counts every non-intercept term in the full model, k1 the single
        # predictor in the base model, so k2 - k1 is the number of terms step 2 added.
        k1, k2 = 1, len(full.params) - 1
        if k2 > k1 and n_e7 - k2 - 1 > 0:
            f_stat = (delta_r2 / (k2 - k1)) / ((1 - full.rsquared) / (n_e7 - k2 - 1))
            p_inc = float(1 - f_dist.cdf(f_stat, k2 - k1, n_e7 - k2 - 1))
    else:
        E7_OK = False
        print(f"  E7 NOT RUN: only {n_e7} complete cases for {len(_have)} facet(s).")

In [ ]:
# === DO (20 of 24): E7, read the models ===
# READING THESE LINES: an R2 on step 1 in the neighbourhood of .47 squared, which is
# about .22, is what the 8-item scale should deliver. A step-1 R2 near .7 would mean
# punitiveness_full got in somehow (.86 squared is .73) -- stop and check the column
# before reading anything else.
if E7_OK and base is not None:
    print(f"E7 complete cases: n = {n_e7}   (smaller than 948: only participants who")
    print("  wrote a substantive backward answer have a facet score)")
    print(f"\nStep 1 (punitiveness, 8 items only): R2 = {base.rsquared:.4f}")
    print(f"Step 2 (+ facets + n_words):         R2 = {full.rsquared:.4f}")
    print(f"Delta R2 = {delta_r2:.4f}, F({k2-k1},{n_e7-k2-1}) = {num(f_stat)}, "
          f"p = {num(p_inc, 5)}")
    # Per-facet coefficients: which facet carries the increment, and in which
    # direction. Only facets that survived into the model are printed.
    print("\nStep-2 facet coefficients:")
    for f in E7_FACETS:
        key = f"ens_{f}"
        if key in full.params.index:
            print(f"  {f:16s} beta = {full.params[key]:+.4f}   "
                  f"p = {full.pvalues[key]:.4f}")
else:
    print("E7 primary: NOT RUN. Reported as not run, never as a null result.")

# r2_base_punit8 is named for the scale it used, so the 8-item restriction is legible
# in the exported CSV and not only in this cell. Shortening it to r2_base would put the
# one number that can invalidate E7 into a column whose name no longer says which
# scale produced it.
e7_df = pd.DataFrame([{
    "test": "E7", "outcome": "Sentence1_z",
    "r2_base_punit8": base.rsquared if base is not None else np.nan,
    "r2_full": full.rsquared if full is not None else np.nan,
    "delta_r2": delta_r2, "F": f_stat, "p": p_inc, "n": n_e7, "family": FAMILY,
    "status": "computed" if (E7_OK and base is not None) else "not run",
    "rests_on": rests(*E7_FACETS)}])

In [ ]:
# === DO (21 of 24): E7 secondary -- does the language reach a non-sentencing outcome? ===
# Prereg 6.7's last sentence on the gate: "We also report whether revenge and
# suffering language predict the shark pain rating and the sentence." Shark_3 is the
# pain rating from the shark judgment task -- an outcome with nothing to do with
# sentencing -- so an association there is evidence that the language is tracking a
# punitive DISPOSITION rather than answering the sentencing question.
#
# Computed at the backward site, which carries one row per participant, so the
# survey column already sitting on those rows is a person-level value and nobody is
# entered more than once.
e7_secondary = []
if SURVEY_OK and "Shark_3" in e7_site.columns:
    for f in ["revenge", "suffering"]:
        col = f"ens_{f}"
        if col not in e7_site.columns:
            continue
        mm = e7_site[[col, "Shark_3"]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(mm) > 2 and mm[col].std() > 0 and mm["Shark_3"].std() > 0:
            r_s, lo_s2, hi_s2 = boot_ci(lambda a, b: pearsonr(a, b)[0],
                                        mm[col].values, mm["Shark_3"].values)
            e7_secondary.append({"test": "E7_secondary", "predictor": f,
                                 "outcome": "Shark_3", "r": r_s, "lo": lo_s2,
                                 "hi": hi_s2, "n": len(mm), "family": FAMILY,
                                 "status": "computed", "rests_on": rests(f)})
e7_secondary_df = pd.DataFrame(e7_secondary)
print("=== E7 secondary (revenge / suffering language -> Shark_3) ===")
print(e7_secondary_df.to_string(index=False,
      na_rep="--", float_format=lambda x: f"{x:.3f}")
      if len(e7_secondary_df) else "  (not run -- no Shark_3 column, or no variance)")

In [ ]:
# === DO (22 of 24): the one family, part 1 -- collect E3, E4 and E5 ===
# WHAT THIS BLOCK DOES: takes one p-value from every hypothesis test in E3, E4, E5,
# E6 and E7, and corrects them TOGETHER with Benjamini-Hochberg at alpha = .05. This
# is the step that fixes what any E-result is allowed to claim.
#
# ONE SLOT PER HYPOTHESIS TEST, NOT ONE PER NUMBER. E3 contributes one row per item x
# facet. E4 contributes ONE, the split-call estimate. E5 contributes one, the
# ensemble text-side estimate. E6 contributes one per facet. E7 contributes the
# incremental F plus each secondary correlation. E4's word-list, human-gold,
# survey-reference, halo and embedding rows are corroboration, reference and
# diagnostic quantities rather than additional hypothesis tests, so they are reported
# in full in e4_df and consume no slot -- charging them one would penalize the
# analysis for having checked its own work.
expl = []

# --- E3: one slot per item x facet -----------------------------------------
# Both Q333 and Q1 enter, which is the price of testing them separately rather than
# pooling, and it is the right price for a within-study replication.
for _, r in e3_df.iterrows():
    expl.append({"test": f"E3_{r['item']}_{r['facet']}", "block": "E3",
                 "p": r["p"], "effect": r["cohens_d"], "rests_on": r["rests_on"]})

# --- E4: exactly one slot, and it is the SPLIT-CALL estimate ---------------
# Selected by the estimate's NAME rather than by its position, so a reordering of
# e4_df can never swap the joint-call number into the family by accident.
_row_split = e4_df[e4_df["estimate"] == "desert_x_revenge_SPLIT_CALL"]
if len(_row_split):
    rr, nn = float(_row_split["r"].iloc[0]), float(_row_split["n"].iloc[0])
    expl.append({"test": "E4_desert_x_revenge_split", "block": "E4",
                 "p": _r_pvalue(rr, nn), "effect": rr,
                 "rests_on": _row_split["rests_on"].iloc[0]})

# --- E5: one slot, and only the text-side ensemble row ---------------------
# The survey side is standing H6 evidence outside this family and the gold rows are
# verification. The FACET is descriptive-only; the TEST is corrected here regardless.
_row_e5 = e5_df[e5_df["estimate"] == "ENSEMBLE_desert_x_proportionality(text_side)"]
if len(_row_e5):
    expl.append({"test": "E5_H6_desert_x_proportionality_text", "block": "E5_H6",
                 "p": float(_row_e5["p"].iloc[0]),
                 "effect": float(_row_e5["value"].iloc[0]),
                 "rests_on": _row_e5["rests_on"].iloc[0]})

In [ ]:
# === DO (22 of 24, cont.): collect E6 and E7, then correct the whole family ===
# --- E6: one slot per facet ------------------------------------------------
# Nine facets means up to nine slots, the largest single contribution to the family.
# That is the correct accounting: E6 makes a directional claim about each facet
# separately, and the descriptive-only facets make it too.
for _, r in e6_df.iterrows():
    expl.append({"test": f"E6_delta_{r['facet']}", "block": "E6", "p": r["p"],
                 "effect": r["mean_delta_bw_minus_fw"], "rests_on": r["rests_on"]})

# --- E7: the incremental F, plus one slot per secondary correlation --------
expl.append({"test": "E7_sentence_delta_r2", "block": "E7",
             "p": float(e7_df["p"].iloc[0]), "effect": float(e7_df["delta_r2"].iloc[0]),
             "rests_on": e7_df["rests_on"].iloc[0]})
for _, r in e7_secondary_df.iterrows():
    expl.append({"test": f"E7sec_{r['predictor']}_{r['outcome']}", "block": "E7",
                 "p": _r_pvalue(r["r"], r["n"]), "effect": r["r"],
                 "rests_on": r["rests_on"]})

# A test that could not produce a p -- a degenerate facet, a missing column, a
# zero-variance difference -- leaves the family here rather than entering it with a
# fabricated number. That makes the family SIZE an observed result of this corpus
# rather than a fixed count, which is why it is printed below: a reader needs to know
# how many tests the correction was actually applied across.
fdr_df = pd.DataFrame(expl)
DROPPED = fdr_df[fdr_df["p"].isna()]["test"].tolist() if len(fdr_df) else []
fdr_df = fdr_df.dropna(subset=["p"]).reset_index(drop=True)
fdr_df["family"] = FAMILY
if len(fdr_df):
    # Benjamini-Hochberg at alpha = .05 (prereg 6.1 uses BH wherever a set of
    # related tests is corrected). BH controls the expected proportion of
    # false discoveries among the discoveries, which is the right criterion for an
    # exploratory family: Bonferroni's family-wise error rate would be a stricter
    # promise than anything in E3-E7 is making.
    rej, p_adj, _, _ = multipletests(fdr_df["p"].values, alpha=0.05, method="fdr_bh")
    fdr_df["p_bh"] = p_adj
    fdr_df["reject_h0_bh"] = rej
    fdr_df["status"] = "computed"

In [ ]:
# === DO (23 of 24): read the corrected family ===
# READING IT: p_bh is the number to quote, never the raw p. reject_h0_bh = True means
# the test survived correction WITHIN AN EXPLORATORY FAMILY -- a lead worth
# following, not a confirmed effect. An empty table means no block produced a usable
# p, which is a pipeline failure rather than a null result: check E4's separate-call
# warning and E6's variance guards first.
print("=== BH-FDR EXPLORATORY FAMILY: E3, E4, E5, E6, E7 -- ONE family (6.7) ===")
if len(fdr_df):
    print(fdr_df[["test", "block", "effect", "p", "p_bh", "reject_h0_bh"]].to_string(
        index=False, na_rep="--", float_format=lambda x: f"{x:.4f}"))
else:
    print("  EMPTY. No block produced a usable p-value. That is a pipeline failure,")
    print("  not a null result.")
print(f"\nFamily size (p-values corrected together): {len(fdr_df)}")
_present = sorted(fdr_df["block"].unique().tolist()) if len(fdr_df) else []
print(f"Blocks represented: {_present if _present else '[]'}")
for b in BLOCKS:
    # A block missing from the family is a block whose tests never ran, and the
    # correction is then over-optimistic by exactly that much. Named one by one so
    # the absence cannot be missed.
    print(f"  {b:6s} " + ("in the family" if b in _present else
                          "ABSENT -- its tests did not run, so the correction below "
                          "covers less than the design intends"))
if DROPPED:
    print("\nTests that produced no p and therefore left the family: "
          + ", ".join(DROPPED))

In [ ]:
# === DO (23 of 24, cont.): what the correction does and does not settle ===
print("NOTHING in E3 to E7 is confirmatory (prereg 2.4), whatever the gate returned")
print("and whatever survives correction here. 'Gate-backed' is a statement about a")
print("FACET's validation standing, never about a TEST's confirmatory status.")
print("\nOUTSIDE this family: the prereg 6.6 non-language exploratory analyses --")
print("E1, E2, the factor structure of the correlates, the sentence change and the")
print("political moderators. Prereg 6.6 leaves those uncorrected in as many words:")
print('  "These analyses are exploratory and are not corrected as a confirmatory')
print('   family."')
print("\nINSIDE it, explicitly: E5 and E6, and every test produced by a")
print("descriptive-only facet (censure, moral_balance, proportionality). Holding any")
print("of them out would understate the multiplicity the analysis actually incurs.")
# The open team decision that changes the size of this family, stated where the size
# is printed rather than in a document nobody has open.
print("\n[TEAM DECISION -- MEASUREMENT_DESIGN 11.12]: the analysis unit and nesting")
print("for E4 to E6 (participant aggregates, mixed models with random intercepts, or")
print("item-level with clustered standard errors). This notebook implements")
print("participant and item-level tables with bootstrap intervals. Swapping in the")
print("ratified nesting also changes how many p-values this one family holds.")

In [ ]:
# === DO (24 of 24): stack every block, and stamp the gate standing on each row ===
# The blocks report deliberately different quantities -- E3 reports Cohen's d, E4 and
# E5 report correlations, E6 reports a mean difference, E7 reports R2 -- so the
# stacked table is SPARSE BY CONSTRUCTION. A blank cell means "this block does not
# report that quantity", never "missing data". The `block` column is what makes the
# stack readable.
e3_e7_results = pd.concat([
    e3_df.assign(block="E3"),
    e4_df.assign(block="E4"),
    e5_df.assign(block="E5_H6"),
    e6_df.assign(block="E6"),
    e7_df.assign(block="E7"),
    e7_secondary_df.assign(block="E7_secondary"),
    fdr_df.assign(block="FDR"),
], ignore_index=True)

# THE STAMP. Every row now says which facets it rests on and how each of them stood
# at the notebook 10 gate. A facet that failed the gate cannot ride into a result
# that reads like a sound one, because the standing is in the row and travels with it
# into any spreadsheet, any sort, any paste into a draft.
_rest = e3_e7_results.get("rests_on", pd.Series([""] * len(e3_e7_results)))
_rest = _rest.fillna("").astype(str)
_lists = _rest.apply(lambda s: [x for x in s.split(",") if x])
e3_e7_results["facet_standing"] = _lists.apply(standing_of)
e3_e7_results["gate_status"] = _lists.apply(gate_status_of)
# One more column so nothing in this file can be read without its standing: the
# family tag is already on the hypothesis-test rows, and this repeats the
# registration's position for the file as a whole.
e3_e7_results["prereg_standing"] = "exploratory (prereg 2.4); FDR within E3-E7 (6.7)"

print(f"Stacked: {len(e3_e7_results)} rows across "
      f"{e3_e7_results['block'].nunique()} blocks.")
print(e3_e7_results["gate_status"].value_counts().to_string())
_mixed = e3_e7_results[e3_e7_results["gate_status"].str.startswith("MIXED")]
if len(_mixed):
    print(f"\n  {len(_mixed)} row(s) rest on a MIX of gate-backed and")
    print("  descriptive-only facets. Those are not gate-backed results, and the")
    print("  facet_standing column names which facet is which.")

## Save and check

In [ ]:
# === SAVE AND CHECK ===
# Two files leave this notebook. Each is written to disk first and registered with
# done() second, so the manifest can never claim a file that is not there.
#   E3_to_E7_results.csv              every block stacked, each row carrying its
#                                     family tag, its status in words, the facets it
#                                     rests on and how each of those stood at the
#                                     gate.
#   fdr_exploratory_family_E3toE7.csv the ONE corrected family. p_bh is the number to
#                                     quote; the raw p is kept beside it so the size
#                                     of the correction is visible.
# THE CONTRACT ON WHAT LEAVES THIS NOTEBOOK. Neither output is keyed on
# (response_id, item_type) -- both are results tables, one row per estimate -- so the
# corpus key contract checked in LOAD and DO 2 does not apply to them. The contract
# that does apply is the family's: ONE Benjamini-Hochberg slot per hypothesis test. A
# test name appearing twice would enter the correction twice and shift EVERY p_bh in
# the table, not only its own, because BH ranks the whole set together.
assert not fdr_df["test"].duplicated().any(), (
    "the corrected family holds a duplicated test name: "
    + ", ".join(sorted(fdr_df.loc[fdr_df["test"].duplicated(), "test"].tolist()))
    + ". Each hypothesis test takes exactly one BH slot; two would shift every "
      "p_bh in the family.")

e3_e7_results.to_csv("E3_to_E7_results.csv", index=False)
done("E3_to_E7_results.csv", rows=len(e3_e7_results),
     note=f"E3-E7 stacked; one BH-FDR family of {len(fdr_df)}; "
          f"{N_BOOT} bootstrap resamples; gate standing stamped per row")

fdr_df.to_csv("fdr_exploratory_family_E3toE7.csv", index=False)
done("fdr_exploratory_family_E3toE7.csv", rows=len(fdr_df),
     note=f"one exploratory family across {', '.join(BLOCKS)}, BH at alpha .05")

In [ ]:
# === SAVE AND CHECK (cont.): what was produced, and what is still not known ===
print(f"\nNOTEBOOK {NB_ID} DONE -- {NB_NAME}")
print(f"  {NB_ASKS}\n")
print("WHAT WAS PRODUCED")
for b in ["E3", "E4", "E5_H6", "E6", "E7", "E7_secondary", "FDR"]:
    print(f"  {b:14s} {int((e3_e7_results['block'] == b).sum()):3d} rows")
print(f"  corrected family size: {len(fdr_df)}"
      + (f", of which {int(fdr_df['reject_h0_bh'].sum())} survive BH"
         if len(fdr_df) else ""))
print(f"  gate-backed facets behind the primary rows: "
      f"{', '.join(GATE_BACKED_FACETS) if GATE_BACKED_FACETS else 'none'}")

print("\nWHAT IS STILL NOT KNOWN")
if not E4_PRIMARY_OK:
    print("  E4's PRIMARY estimate did not run: the separate-call columns are not in")
    print("  ensemble_scores.csv, so the study's central test is missing from the")
    print("  family above. Re-run notebook 06 with the split-call block, then 07.")
print("  The word-list corroborator is preregistered and it was run. Read it beside")
print("  its prevalence and its joint-firing count: at the base rates this corpus")
print("  carries it can neither corroborate E4 nor refute it, and that is an")
print("  observed limitation to report with the counts, not a null.")
if not HUMAN_GOLD:
    print("  No human has read these responses anywhere in this chain, so the added")
    print("  human corroborator reads 'not run' for E4 and E5. Prereg 6.7 prescribes")
    print("  no human-coding step, so this run is the registered design exactly; the")
    print("  gap it leaves is the design's, and it belongs in the limitations.")
print("  Nothing here is confirmatory. All five blocks are one exploratory family")
print("  (prereg 2.4, 6.7), and surviving BH inside it makes a result a lead worth")
print("  following, never a confirmed effect.")
print("\n  next: notebook 12, which collects these two files into the OSF bundle and")
print("  writes what 02_nlp_integration.Rmd reads.")